# deblur-vs-filter on Kaggle

Classical pseudo-inverse filter vs DeblurGAN-v2 for motion-blurred text: OCR quality (CER), time and energy.

Settings: Accelerator = GPU T4, Internet = On.

What it does:
1. makes 35 sharp text images 512×256 (5 validation + 30 test) and blurs each one under 48 conditions
   (4 kinds of motion × L = 10/20/32 px × noise σ = 0/2/5/10), 1440 test images in total
2. picks α² on the validation images, runs 6 methods on all test images, Tesseract OCR → CER, WER, PSNR, SSIM
3. measures time and energy (CodeCarbon) on 30 images, filters on CPU, networks on CPU and GPU
4. statistics per text (n = 30), figures, all numbers used in the paper

Save & Run All takes about 1 hour. Results are in the Output tab: `results_meics.zip`.

## 1. Settings

In [ ]:
# settings
RUN_SYNTHETIC = True     # main experiment: 30 test + 5 validation texts x 48 conditions
N_TEST, N_VAL = 30, 5    # with other numbers the md5 check in step 5 will not match (fine for a quick try)

import os
WORK = '/kaggle/working/meics'
for d in ('deblur', 'scripts', 'texts'):
    os.makedirs(f'{WORK}/{d}', exist_ok=True)
%cd {WORK}

## 2. Code
These cells just save the package and scripts to disk (same files as in the repo).

`deblur/__init__.py`

In [ ]:
%%writefile deblur/__init__.py
"""Blur models, text images, classical filters (1D and 2D), DeblurGAN-v2 wrapper, metrics, time and energy."""

`deblur/degradation.py` - blur models: sinc (formula 1), box, line at 8°, hand shake + gamma + JPEG

In [ ]:
%%writefile deblur/degradation.py
"""Blur models: how a sharp text image f becomes a blurred image g.

Main model (horizontal motion), formula (1) in the paper:
    H(wx, wy) = sin(beta * k) / (beta * k),  k = column index from the centre of the spectrum
Blur length in pixels: L = beta * n / pi, so beta = pi * L / n (n = image width).
For n = 512 and beta = 0.194, L is about 31.6 px.

Four blur models (argument `model` of degrade() at the bottom):
    "sinc"   - multiply the spectrum by H, exactly formula (1)
    "box"    - same horizontal motion, but each pixel = mean of L neighbours
    "angle8" - straight motion at 8 degrees
    "shake"  - hand shake: random curved path + phone-like processing (gamma, noise, JPEG)
"""
from __future__ import annotations

import numpy as np


# ---- 1. sinc model

def freq_index(width: int) -> np.ndarray:
    """Column numbers from the centre: -n//2 ... n - n//2 - 1 (zero in the middle, like after fftshift)."""
    return np.arange(width) - (width // 2)


def sinc_mtf(shape: tuple[int, int], beta: float) -> np.ndarray:
    """H = sin(beta*k)/(beta*k), centred (fftshift)."""
    h, w = shape
    k = freq_index(w).astype(np.float64)
    # np.sinc(x) is sin(pi x)/(pi x), so divide by pi; at k = 0 it returns 1
    hk = np.sinc(beta * k / np.pi)
    # horizontal motion: H depends on wx only, same row for every image row
    return np.broadcast_to(hk[None, :], (h, w)).copy()


def beta_to_length(beta: float, width: int) -> float:
    return beta * width / np.pi


def length_to_beta(length_px: float, width: int) -> float:
    return np.pi * length_px / width


def blur_sinc(img: np.ndarray, beta: float) -> np.ndarray:
    """G = H * F in the frequency domain (circular convolution)."""
    f = img.astype(np.float64)
    spec = np.fft.fftshift(np.fft.fft2(f))
    # imaginary part is only rounding noise
    g = np.real(np.fft.ifft2(np.fft.ifftshift(spec * sinc_mtf(f.shape, beta))))
    return g


# ---- 2. blur in pixels (convolution with the point spread function)

def blur_box(img: np.ndarray, length_px: int) -> np.ndarray:
    """Horizontal box kernel of length L with mirrored edges.
    Not exactly the filter's model, so it is a harder test for the methods."""
    import cv2
    L = max(1, int(round(length_px)))
    # L values of 1/L: each pixel becomes the mean of L neighbours in its row
    kernel = np.full((1, L), 1.0 / L, dtype=np.float64)
    # mirrored edges (the sinc model wraps around instead)
    return cv2.filter2D(img.astype(np.float64), -1, kernel, borderType=cv2.BORDER_REFLECT)


def motion_kernel(length_px: float, angle_deg: float) -> np.ndarray:
    """Straight line of length L at an angle (anti-aliased), like fspecial('motion', L, theta)."""
    import cv2
    L = max(1.0, float(length_px))
    size = int(np.ceil(L)) + 2
    size += (size + 1) % 2                     # odd size, so there is a centre pixel
    scale = 8                                  # draw 8x bigger, then shrink = anti-aliasing
    big = np.zeros((size * scale, size * scale), np.float32)
    c = (size * scale - 1) / 2.0
    # line ends: L/2 from the centre each way; minus because image y goes down
    dx = np.cos(np.deg2rad(angle_deg)) * (L - 1) * scale / 2.0
    dy = -np.sin(np.deg2rad(angle_deg)) * (L - 1) * scale / 2.0
    cv2.line(big, (int(round(c - dx)), int(round(c - dy))), (int(round(c + dx)), int(round(c + dy))),
             1.0, thickness=scale, lineType=cv2.LINE_AA)
    k = cv2.resize(big, (size, size), interpolation=cv2.INTER_AREA).astype(np.float64)
    return k / k.sum()                          # kernel sums to 1


def blur_angle(img: np.ndarray, length_px: float, angle_deg: float) -> np.ndarray:
    """Motion at an angle: the horizontal filter model is no longer exact."""
    import cv2
    return cv2.filter2D(img.astype(np.float64), -1, motion_kernel(length_px, angle_deg),
                        borderType=cv2.BORDER_REFLECT)


# ---- 3. hand shake: random curved path

def shake_kernel(length_px: float, rng: np.random.Generator, n_steps: int = 400,
                 anxiety: float = 0.005) -> np.ndarray:
    """Kernel of a random curved camera path.
    The direction changes smoothly with rare sharp jerks; the path is scaled so that its
    largest span is L px. It is not a line, so no classical filter knows its shape."""
    # position as a complex number x + jy
    L = max(2.0, float(length_px))
    v = np.exp(1j * rng.uniform(0, 2 * np.pi))       # start direction
    x = np.zeros(n_steps, complex)                   # 400 camera positions during exposure
    for t in range(1, n_steps):
        dv = rng.normal() + 1j * rng.normal()
        if rng.random() < anxiety * 10:            # 5 % chance of a jerk 8x stronger
            dv *= 8
        # mostly keep the old direction, so the path is a smooth curve
        v = 0.995 * v + 0.06 * dv
        v /= max(abs(v), 1e-9)                       # step length stays 1
        x[t] = x[t - 1] + v
    x -= x.mean()
    # scale the larger of the x and y spans to L - 1 px
    span = max(np.ptp(x.real), np.ptp(x.imag), 1e-9)
    x *= (L - 1) / span
    size = int(np.ceil(L)) + 4
    size += (size + 1) % 2
    k = np.zeros((size, size))
    c = (size - 1) / 2.0
    # each path point adds weight 1, split between 4 neighbour pixels (bilinear)
    for z in x:
        px, py = z.real + c, z.imag + c
        i0, j0 = int(np.floor(py)), int(np.floor(px))
        fy, fx = py - i0, px - j0
        for di, wy in ((0, 1 - fy), (1, fy)):
            for dj, wx in ((0, 1 - fx), (1, fx)):
                if 0 <= i0 + di < size and 0 <= j0 + dj < size:
                    k[i0 + di, j0 + dj] += wy * wx
    return k / k.sum()


# ---- 4. phone-like processing (shake only)

def _to_linear(x):
    # sRGB -> linear light (IEC 61966-2-1)
    x = np.clip(x, 0, 255) / 255.0
    return 255.0 * np.where(x <= 0.04045, x / 12.92, ((x + 0.055) / 1.055) ** 2.4)


def _to_srgb(x):
    # linear light -> sRGB (gamma)
    x = np.clip(x, 0, 255) / 255.0
    return 255.0 * np.where(x <= 0.0031308, 12.92 * x, 1.055 * x ** (1 / 2.4) - 0.055)


def camera_like(img: np.ndarray, kernel: np.ndarray, sigma: float, rng: np.random.Generator,
                jpeg_quality: int = 85) -> np.ndarray:
    """Rough smartphone photo: blur in linear light, sRGB gamma, sensor noise, 8 bit, JPEG."""
    import cv2
    # the sensor collects light (linear) while moving
    lin = cv2.filter2D(_to_linear(img.astype(np.float64)), -1, kernel, borderType=cv2.BORDER_REFLECT)
    g = add_noise_and_quantize(_to_srgb(lin), sigma, rng)
    # save as JPEG (quality 85) and read back to get compression artefacts
    ok, buf = cv2.imencode(".jpg", g, [cv2.IMWRITE_JPEG_QUALITY, jpeg_quality])
    return cv2.imdecode(buf, cv2.IMREAD_GRAYSCALE)


def add_noise_and_quantize(g: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Gaussian noise with std sigma (grey levels 0..255), then round to uint8."""
    noisy = g + rng.normal(0.0, sigma, size=g.shape) if sigma > 0 else g
    return np.clip(np.round(noisy), 0, 255).astype(np.uint8)


# ---- 5. main entry

def degrade(img: np.ndarray, model: str, length_px: float, sigma: float,
            rng: np.random.Generator) -> tuple[np.ndarray, float]:
    """Returns (blurred uint8 image, true beta).
    True beta is saved so we can check the cepstrum estimate and run the filter with known beta."""
    width = img.shape[1]
    if model == "sinc":
        beta = length_to_beta(length_px, width)
        g = blur_sinc(img, beta)
    elif model == "box":
        L = int(round(length_px))
        beta = length_to_beta(L, width)
        g = blur_box(img, L)
    elif model.startswith("angle"):          # e.g. "angle8"
        angle = float(model[len("angle"):] or 8)
        # nominal beta = horizontal projection of the motion
        beta = length_to_beta(length_px * np.cos(np.deg2rad(angle)), width)
        g = blur_angle(img, length_px, angle)
    elif model == "shake":
        beta = length_to_beta(length_px, width)   # nominal only, there is no true beta here
        return camera_like(img, shake_kernel(length_px, rng), sigma, rng), beta
    else:
        raise ValueError(f"unknown blur model: {model}")
    # shake adds noise inside camera_like, the others add it here
    return add_noise_and_quantize(g, sigma, rng), beta

`deblur/textgen.py` - draws text on a white background and keeps the true text

In [ ]:
%%writefile deblur/textgen.py
"""Sharp text images with known ground-truth text.

We draw the text ourselves, so we know exactly what is written (needed for CER).
Text comes from texts/holmes_scenes.txt word by word (class Corpus).
WORDS_ENG / WORDS_UKR and random_line() are only a fallback; 01_make_dataset.py always passes a corpus.
"""
from __future__ import annotations

import glob
import os

import numpy as np
from PIL import Image, ImageDraw, ImageFont

# fallback word lists (not used when a corpus is given)
WORDS_ENG = (
    "the of and to in is that for it as was with be by on not he this are or his from at which "
    "but have an they you were her she there been one all we their has would when if so no will "
    "more other time some could these two may first then do any like my now over such our man me "
    "even most made after also did many before must through back years where much your way well "
    "down should because each just those people how too little state good very make world still own "
    "see men work long get here between both life being under never day same another know while "
    "last might us great old year off come since against go came right used take three energy image "
    "signal filter noise model data method result system network value pixel matrix spectrum "
    "frequency camera motion blur restore quality metric error test sample power heat server cloud"
).split()

WORDS_UKR = (
    "і в на що не з до як це та але він вона вони ми ви так вже ще був була було якщо коли де "
    "тут там дуже можна треба рік час день ніч місто людина робота життя слово світ земля вода "
    "сила точка лінія площа сигнал фільтр шум модель дані метод результат система мережа значення "
    "піксель матриця спектр частота камера рух розмиття якість помилка тест зразок енергія тепло "
    "сервер обчислення зображення текст відновлення оцінка параметр функція перетворення"
).split()

# only if matplotlib fonts are missing (on Kaggle they are always there)
FONT_CANDIDATES = [
    "/usr/share/fonts/truetype/liberation/LiberationSerif-Regular.ttf",
    "/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf",
    "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    "C:/Windows/Fonts/times.ttf",
    "C:/Windows/Fonts/arial.ttf",
]


def find_fonts() -> list[str]:
    """DejaVu Serif and Sans shipped with matplotlib: same on every machine, so the images are reproducible."""
    import matplotlib
    d = os.path.join(os.path.dirname(matplotlib.__file__), "mpl-data", "fonts", "ttf")
    found = [os.path.join(d, f) for f in ("DejaVuSerif.ttf", "DejaVuSans.ttf") if os.path.exists(os.path.join(d, f))]
    if not found:
        found = [p for p in FONT_CANDIDATES if os.path.exists(p)]
    return found


def _load_font(path: str | None, size: int):
    if path:
        return ImageFont.truetype(path, size)
    return ImageFont.load_default(size=size)   # Pillow >= 10.1


def random_line(rng: np.random.Generator, words: list[str], n_words: int) -> str:
    # not used in the experiment: random words, sometimes a number
    ws = list(rng.choice(words, size=n_words))
    if rng.random() < 0.5:
        ws.insert(int(rng.integers(0, len(ws))), str(int(rng.integers(10, 9999))))
    return " ".join(ws)


class Corpus:
    """Real text (passages from "The Adventures of Sherlock Holmes"), handed out word by word;
    starts again from the beginning when it runs out."""

    def __init__(self, path: str):
        self.words = open(path, encoding="utf-8").read().split()
        self.i = 0

    def next(self) -> str:
        if self.i == len(self.words):
            print("warning: corpus ended, starting again from the beginning")
        w = self.words[self.i % len(self.words)]
        self.i += 1
        return w

    def peek(self) -> str:
        # look at the next word without taking it
        return self.words[self.i % len(self.words)]


def make_text_image(rng: np.random.Generator, width: int = 512, height: int = 256,
                    lang: str = "eng", fonts: list[str] | None = None,
                    corpus: "Corpus | None" = None) -> tuple[np.ndarray, str]:
    """Black text on white, 4-6 lines. Returns (uint8 HxW image, ground-truth text)."""
    words = WORDS_UKR if lang == "ukr" else WORDS_ENG
    fonts = fonts if fonts is not None else find_fonts()
    # random but reproducible font and size 22-30
    font_path = fonts[int(rng.integers(0, len(fonts)))] if fonts else None
    size = int(rng.integers(22, 31))
    font = _load_font(font_path, size)
    img = Image.new("L", (width, height), color=255)
    draw = ImageDraw.Draw(img)
    margin_x, y = 16, 14
    line_h = int(size * 1.35)
    lines = []
    while y + line_h <= height - 8:
        # add words while the line still fits
        if corpus is not None:
            line = corpus.next()
            while draw.textlength(line + " " + corpus.peek(), font=font) <= width - 2 * margin_x:
                line += " " + corpus.next()
        else:
            n = int(rng.integers(3, 7))
            line = random_line(rng, words, n)
            while draw.textlength(line, font=font) > width - 2 * margin_x and len(line.split()) > 1:
                line = " ".join(line.split()[:-1])
        draw.text((margin_x, y), line, fill=0, font=font)
        lines.append(line)
        y += line_h
    return np.array(img, dtype=np.uint8), "\n".join(lines)

`deblur/classical.py` - 1D pseudo-inverse filter and blur length from the cepstrum

In [ ]:
%%writefile deblur/classical.py
"""Classical restoration, 1D version (as in our earlier paper [2]).

1) pseudo-inverse filter, formula (2):
       W = H* / (|H|^2 + alpha^2),   F^ = W * G,   H = sin(beta*wx)/(beta*wx)
2) blind estimate of the blur length L from the cepstrum, so beta is not set by hand.
"""
from __future__ import annotations

import numpy as np

from .degradation import sinc_mtf, beta_to_length, length_to_beta

# alpha^2 values we try: 1e-8 ... 1, one order of magnitude apart
ALPHA2_GRID = [10.0 ** p for p in range(-8, 1)]   # 1e-8 ... 1


def pseudo_inverse(g: np.ndarray, beta: float, alpha2: float) -> np.ndarray:
    """Pseudo-inverse filter; output clipped to [0, 255]."""
    g = g.astype(np.float64)
    spec = np.fft.fftshift(np.fft.fft2(g))
    H = sinc_mtf(g.shape, beta)
    W = np.conj(H) / (np.abs(H) ** 2 + alpha2)
    f = np.real(np.fft.ifft2(np.fft.ifftshift(spec * W)))
    return np.clip(f, 0.0, 255.0)


def estimate_length_cepstrum(g: np.ndarray, l_min: float = 3.0, l_max: float | None = None) -> float:
    """Blind estimate of the horizontal blur length L (px) from the cepstrum.

    Zeros of sinc(beta*k) repeat every pi/beta, so log|spectrum| has a periodic part
    and its inverse FFT (the cepstrum) has a negative peak at q = L.
    """
    x = g.astype(np.float64)
    x = x - x.mean()                       # remove the big zero frequency
    h, w = x.shape
    # |FFT| of every row, averaged over rows: the blur is the same in each row,
    # averaging keeps it and smooths out letters and noise
    mag = np.abs(np.fft.fft(x, axis=1)).mean(axis=0)
    ceps = np.real(np.fft.ifft(np.log(mag + 1e-8)))
    # deepest minimum between 3 px and a quarter of the width
    if l_max is None:
        l_max = w / 4
    lo, hi = int(np.floor(l_min)), int(np.ceil(l_max))
    seg = ceps[lo:hi + 1]
    i = int(np.argmin(seg))
    q = float(lo + i)
    # parabola through the minimum and its neighbours -> sub-pixel L
    if 0 < i < len(seg) - 1:
        y0, y1, y2 = seg[i - 1], seg[i], seg[i + 1]
        den = y0 - 2 * y1 + y2
        if abs(den) > 1e-12:
            q += 0.5 * (y0 - y2) / den
    return q


def estimate_beta(g: np.ndarray, **kw) -> float:
    return length_to_beta(estimate_length_cepstrum(g, **kw), g.shape[1])


def restore_oracle(g: np.ndarray, beta_true: float, alpha2: float) -> np.ndarray:
    """Filter with the true beta (upper bound for the classical method)."""
    return pseudo_inverse(g, beta_true, alpha2)


def restore_blind(g: np.ndarray, alpha2: float) -> tuple[np.ndarray, float]:
    """Fully automatic: beta from the cepstrum + pseudo-inverse filter."""
    beta = estimate_beta(g)
    return pseudo_inverse(g, beta, alpha2), beta


__all__ = ["ALPHA2_GRID", "pseudo_inverse", "estimate_length_cepstrum", "estimate_beta",
           "restore_oracle", "restore_blind", "beta_to_length", "length_to_beta"]

`deblur/classical2d.py` - 2D filter: L and angle from the 2D cepstrum, mirror padding

In [ ]:
%%writefile deblur/classical2d.py
"""Classical restoration, 2D version: straight motion in any direction.

Differences from the 1D version (classical.py):
  1) the blur is a line of length L at any angle theta; both are found from the 2D cepstrum
  2) H is the FFT of that line kernel (like fspecial('motion', L, theta)), not the sinc formula;
     the filter itself is the same: W = H* / (|H|^2 + alpha^2)
  3) image edges are mirror-padded before filtering to reduce ringing

Why the cepstrum gives both L and theta: the spectrum of a line is sinc along the motion and
flat across it, so ln|G| is periodic along theta. Its 2D cepstrum IFFT2{ln|G|} has negative
peaks at (+-L*cos(theta), -+L*sin(theta)).
"""
from __future__ import annotations

import numpy as np

from .degradation import motion_kernel


def _hann2d(shape):
    # 2D Hann window: 1 in the centre, 0 at the edges
    return np.outer(np.hanning(shape[0]), np.hanning(shape[1]))


def cepstrum2d(g: np.ndarray) -> np.ndarray:
    # c = IFFT2{ ln|FFT2{g}| }, centred
    x = g.astype(np.float64)
    # the window removes the bright cross that sharp image borders leave in the spectrum
    x = (x - x.mean()) * _hann2d(x.shape)
    mag = np.abs(np.fft.fft2(x))
    return np.fft.fftshift(np.real(np.fft.ifft2(np.log(mag + 1e-8))))


def estimate_motion_2d(g: np.ndarray, l_min: float = 4.0, l_max: float | None = None):
    """Returns (L in px, theta in degrees, 0 <= theta < 180)."""
    c = cepstrum2d(g)
    h, w = c.shape
    cy, cx = h // 2, w // 2
    if l_max is None:
        l_max = min(h, w) / 4                       # 64 px for 512x256
    yy, xx = np.mgrid[0:h, 0:w]
    r = np.hypot(yy - cy, xx - cx)
    # search only in a ring: closer than 4 px there is always a big hill around the centre
    mask = (r >= l_min) & (r <= l_max)
    # this line changes nothing in practice (r >= l_min already removes these points);
    # the horizontal axis itself stays in the search, that is where horizontal blur peaks
    mask &=(yy != cy) | (np.abs(xx - cx) >= l_min)
    cm = np.where(mask, c, np.inf)
    iy, ix = np.unravel_index(np.argmin(cm), cm.shape)
    dy, dx = iy - cy, ix - cx
    # sub-pixel refinement along each axis (parabola through 3 points)
    def refine(v, i, axis):
        if axis == 0 and 0 < iy < h - 1:
            y0, y1, y2 = c[iy - 1, ix], c[iy, ix], c[iy + 1, ix]
        elif axis == 1 and 0 < ix < w - 1:
            y0, y1, y2 = c[iy, ix - 1], c[iy, ix], c[iy, ix + 1]
        else:
            return float(v)
        den = y0 - 2 * y1 + y2
        return float(v) + (0.5 * (y0 - y2) / den if abs(den) > 1e-12 else 0.0)
    dy, dx = refine(dy, iy, 0), refine(dx, ix, 1)
    # L = distance to the peak, theta = its direction (image y goes down; theta and theta+180 are the same line)
    L = float(np.hypot(dx, dy))
    theta = float(np.degrees(np.arctan2(-dy, dx)) % 180.0)
    return L, theta


def psf_to_otf(psf: np.ndarray, shape) -> np.ndarray:
    """Small kernel -> transfer function of the image size, kernel centre moved to (0, 0) (like psf2otf)."""
    pad = np.zeros(shape, np.float64)
    kh, kw = psf.shape
    pad[:kh, :kw] = psf
    pad = np.roll(pad, (-(kh // 2), -(kw // 2)), axis=(0, 1))
    return np.fft.fft2(pad)


def deconv2d(g: np.ndarray, length_px: float, theta_deg: float, alpha2: float) -> np.ndarray:
    """2D pseudo-inverse filter with mirror padding."""
    g = g.astype(np.float64)
    m = int(np.ceil(length_px)) + 8
    # FFT treats the image as periodic; a mirrored border makes the wrap-around smooth (less ringing)
    gp = np.pad(g, m, mode="reflect")
    H = psf_to_otf(motion_kernel(length_px, theta_deg), gp.shape)
    W = np.conj(H) / (np.abs(H) ** 2 + alpha2)
    # filter, then cut the padding off
    f = np.real(np.fft.ifft2(np.fft.fft2(gp) * W))[m:-m, m:-m]
    return np.clip(f, 0.0, 255.0)


def restore_blind_2d(g: np.ndarray, alpha2: float, **kw):
    # fully automatic: L and theta from the cepstrum, then the filter
    L, th = estimate_motion_2d(g, **kw)
    return deconv2d(g, L, th, alpha2), L, th


__all__ = ["cepstrum2d", "estimate_motion_2d", "psf_to_otf", "deconv2d", "restore_blind_2d"]

`deblur/metrics.py` - CER, WER, PSNR, SSIM, Tesseract

In [ ]:
%%writefile deblur/metrics.py
"""Quality metrics.

  CER  - character error rate, the main metric
  WER  - the same for words
  PSNR - restored vs sharp image; used to pick alpha^2 on validation texts
  SSIM - structural similarity, saved in quality.csv
OCR is Tesseract through pytesseract.
"""
from __future__ import annotations

import re

import numpy as np
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

# edit (Levenshtein) distance; rapidfuzz is fast
try:
    from rapidfuzz.distance import Levenshtein as _Lev

    def _edit(a, b):
        return _Lev.distance(a, b)
except ImportError:  # slow fallback, plain dynamic programming
    def _edit(a, b):
        prev = list(range(len(b) + 1))
        for i, ca in enumerate(a, 1):
            cur = [i]
            for j, cb in enumerate(b, 1):
                cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
            prev = cur
        return prev[-1]


def psnr(ref: np.ndarray, x: np.ndarray) -> float:
    # 10*log10(255^2 / MSE), higher = closer to the sharp image
    return float(peak_signal_noise_ratio(ref.astype(np.float64), x.astype(np.float64), data_range=255))


def ssim(ref: np.ndarray, x: np.ndarray) -> float:
    # 1 = identical images
    return float(structural_similarity(ref.astype(np.float64), x.astype(np.float64), data_range=255))


def normalize_text(s: str) -> str:
    # same dashes and quotes, single spaces, lower case, so these are not counted as errors
    s = s.replace("–", "-").replace("–", "-")
    s = s.replace("“", '"').replace("”", '"').replace("„", '"').replace("‘", "'").replace("’", "'")
    return re.sub(r"\s+", " ", s).strip().lower()


def cer(ref_text: str, hyp_text: str) -> float:
    # edit distance / length of the true text; 0 = perfect, can be > 1 if OCR adds a lot of junk
    r, h = normalize_text(ref_text), normalize_text(hyp_text)
    return _edit(r, h) / max(1, len(r))


def wer(ref_text: str, hyp_text: str) -> float:
    r, h = normalize_text(ref_text).split(), normalize_text(hyp_text).split()
    return _edit(r, h) / max(1, len(r))


def ocr(img: np.ndarray, lang: str = "eng") -> str:
    # --psm 6: treat the image as one block of text
    import pytesseract
    x = np.clip(img, 0, 255).astype(np.uint8)
    return pytesseract.image_to_string(x, lang=lang, config="--psm 6")


def ocr_confidence(img: np.ndarray, lang: str = "eng") -> float:
    """Mean Tesseract confidence (0-100), for images without ground-truth text."""
    import pytesseract
    x = np.clip(img, 0, 255).astype(np.uint8)
    d = pytesseract.image_to_data(x, lang=lang, config="--psm 6", output_type=pytesseract.Output.DICT)
    conf = [float(c) for c, t in zip(d["conf"], d["text"]) if t.strip() and float(c) >= 0]
    return float(np.mean(conf)) if conf else 0.0

`deblur/energy.py` - time and energy (CodeCarbon)

In [ ]:
%%writefile deblur/energy.py
"""Time and energy.

Time: a stopwatch (time.perf_counter) around one call.

Energy: CodeCarbon (https://codecarbon.io).
  - NVIDIA GPU is measured directly (NVML)
  - CPU is measured only with Intel RAPL; otherwise CodeCarbon estimates it from TDP and load,
    so the energy is nearly proportional to time. The mode is saved as cpu_tracking.
    On Kaggle it is "cpu_load", i.e. an estimate.
  - tracking_mode="machine": energy of the whole machine (CPU + GPU + RAM), gross_J_per_img
We also measure the idle machine and subtract it (net_J_per_img, saved in efficiency.csv).
"""
from __future__ import annotations

import time
import warnings

import numpy as np


def _tracker(project: str):
    # polls power every second, writes no files
    from codecarbon import OfflineEmissionsTracker
    return OfflineEmissionsTracker(project_name=project, country_iso_code="UKR",
                                   measure_power_secs=1, save_to_file=False,
                                   log_level="error", tracking_mode="machine")


def _energy_joules(tracker) -> tuple[float, dict]:
    # CodeCarbon gives kWh; 1 kWh = 3.6e6 J
    d = tracker.final_emissions_data
    info = {"cpu_model": getattr(d, "cpu_model", ""), "gpu_model": getattr(d, "gpu_model", ""),
            "cpu_J": float(getattr(d, "cpu_energy", 0.0)) * 3.6e6,
            "gpu_J": float(getattr(d, "gpu_energy", 0.0)) * 3.6e6,
            "ram_J": float(getattr(d, "ram_energy", 0.0)) * 3.6e6}
    # how CPU power was obtained: "intel_rapl" = measured, "cpu_load"/"constant" = estimated
    info["cpu_tracking"] = ""
    for hw in getattr(tracker, "_hardware", []):
        if type(hw).__name__ == "CPU":
            info["cpu_tracking"] = str(getattr(hw, "_mode", ""))
    return float(d.energy_consumed) * 3.6e6, info


def measure_idle_power(seconds: float = 20.0) -> float:
    """Mean idle power of the machine, W (to subtract later)."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        tr = _tracker("idle")
        tr.start()
        t0 = time.perf_counter()
        time.sleep(seconds)
        tr.stop()
        dt = time.perf_counter() - t0
    e, _ = _energy_joules(tr)
    return e / dt


def time_per_item(fn, items, warmup: int = 2, sync=None) -> np.ndarray:
    # seconds per image; the first 2 calls are warm-up (library loading, GPU init) and not counted
    for it in items[:warmup]:
        fn(it)
    out = []
    for it in items:
        t0 = time.perf_counter()
        fn(it)
        if sync:
            sync()        # GPU: wait until it has really finished
        out.append(time.perf_counter() - t0)
    return np.array(out)


def energy_per_item(fn, items, min_seconds: float = 30.0, idle_power_W: float = 0.0,
                    project: str = "method") -> dict:
    """Runs fn over items in a loop for at least min_seconds and returns J per image.
    One 10 ms call is too short for a meter that polls once a second, hence the loop."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        tr = _tracker(project)
        tr.start()
        n, t0 = 0, time.perf_counter()
        while n < len(items) or time.perf_counter() - t0 < min_seconds:
            fn(items[n % len(items)])
            n += 1
        tr.stop()
        dt = time.perf_counter() - t0
    e, info = _energy_joules(tr)
    return {"n_runs": n, "seconds": dt, "gross_J_per_img": e / n,
            "net_J_per_img": max(0.0, (e - idle_power_W * dt) / n),       # minus idle
            "avg_power_W": e / dt, **info}

`deblur/deblurgan.py` - wrapper for the authors' DeblurGAN-v2 code and weights (no training)

In [ ]:
%%writefile deblur/deblurgan.py
"""DeblurGAN-v2 wrapper.

Official code: https://github.com/VITA-Group/DeblurGANv2 (cloned by the notebook),
authors' weights: fpn_inception.h5 (heavy, 61 M params) and fpn_mobilenet.h5 (light, 3.3 M).
We do not train the network, only run it as published.

Differences from their predict.py (no effect on the output):
  - runs on CPU and GPU (the original always calls .cuda())
  - does not download ImageNet backbone weights (they are overwritten by the .h5 anyway)
  - takes and returns a grayscale image, copied to 3 channels
model.train(True) is kept as in the original: normalisation uses statistics of the current image.
"""
from __future__ import annotations

import functools
import os
import sys

import numpy as np


class DeblurGANv2:
    def __init__(self, repo_dir: str, weights_path: str, backbone: str = "fpn_inception",
                 device: str | None = None):
        import torch
        import torch.nn as nn

        # so we can import the authors' models
        repo_dir = os.path.abspath(repo_dir)
        if repo_dir not in sys.path:
            sys.path.insert(0, repo_dir)
        # InstanceNorm, as in the authors' config
        norm_layer = functools.partial(nn.InstanceNorm2d, affine=False, track_running_stats=True)

        if backbone == "fpn_inception":
            # heavy: Inception-ResNet-v2 backbone + FPN
            import models.fpn_inception as fi
            from pretrainedmodels import inceptionresnetv2 as _irv2
            # build the backbone without ImageNet weights, the .h5 replaces them anyway
            fi.inceptionresnetv2 = lambda num_classes=1000, pretrained="imagenet": _irv2(
                num_classes=num_classes, pretrained=None)
            model = fi.FPNInception(norm_layer=norm_layer)
        elif backbone == "fpn_mobilenet":
            # light: MobileNet-v2 backbone
            from models.fpn_mobilenet import FPNMobileNet
            model = FPNMobileNet(norm_layer=norm_layer, pretrained=False)
        else:
            raise ValueError(backbone)

        if weights_path:
            # authors' weights; drop the "module." prefix left by multi-GPU training
            ckpt = torch.load(weights_path, map_location="cpu", weights_only=False)
            state = ckpt["model"] if "model" in ckpt else ckpt
            state = {k[len("module."):] if k.startswith("module.") else k: v for k, v in state.items()}
            model.load_state_dict(state)

        self.torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)
        self.model.train(True)
        self.backbone = backbone

    @property
    def n_params(self) -> int:
        return int(sum(p.numel() for p in self.model.parameters()))

    def __call__(self, gray: np.ndarray) -> np.ndarray:
        torch = self.torch
        # trained on colour photos: copy gray to 3 channels, scale 0..255 to -1..1
        x = np.repeat(gray.astype(np.float32)[..., None], 3, axis=2) / 255.0
        x = (x - 0.5) / 0.5
        # sizes must be multiples of 32, pad with zeros
        h, w, _ = x.shape
        H = (h // 32 + 1) * 32
        W = (w // 32 + 1) * 32
        x = np.pad(x, ((0, H - h), (0, W - w), (0, 0)), mode="constant")
        # (1, 3, H, W)
        t = torch.from_numpy(np.transpose(x, (2, 0, 1))[None]).to(self.device)
        with torch.no_grad():
            y = self.model(t)
        if self.device.startswith("cuda"):
            torch.cuda.synchronize()             # wait for the GPU, for honest timing
        y = y[0].detach().cpu().float().numpy()
        # back to 0..255, cut the padding, average the 3 channels
        y = (np.transpose(y, (1, 2, 0)) + 1.0) / 2.0 * 255.0
        return np.clip(y[:h, :w, :].mean(axis=2), 0, 255)

`texts/holmes_scenes.txt` - text for the images (passages from "The Adventures of Sherlock Holmes")

In [ ]:
%%writefile texts/holmes_scenes.txt
I had called upon my friend, Mr. Sherlock Holmes, one day in the autumn of last year and found him in deep conversation with a very stout, florid-faced, elderly gentleman with fiery red hair. With an apology for my intrusion, I was about to withdraw when Holmes pulled me abruptly into the room and closed the door behind me. We travelled by the Underground as far as Aldersgate; and a short walk took us to Saxe-Coburg Square, the scene of the singular story which we had listened to in the morning. It was a poky, little, shabby-genteel place, where four lines of dingy two-storied brick houses looked out into a small railed-in enclosure, where a lawn of weedy grass and a few clumps of faded laurel bushes made a hard fight against a smoke-laden and uncongenial atmosphere. Three gilt balls and a brown board with "JABEZ WILSON" in white letters, upon a corner house, announced the place where our red-headed client carried on his business. Sherlock Holmes stopped in front of it with his head on one side and looked it all over, with his eyes shining brightly between puckered lids. Then he walked slowly up the street, and then down again to the corner, still looking keenly at the houses. Finally he returned to the pawnbroker's, and, having thumped vigorously upon the pavement with his stick two or three times, he went up to the door and knocked. It was instantly opened by a bright-looking, clean-shaven young fellow, who asked him to step in. The road in which we found ourselves as we turned round the corner from the retired Saxe-Coburg Square presented as great a contrast to it as the front of a picture does to the back. It was one of the main arteries which conveyed the traffic of the City to the north and west. The roadway was blocked with the immense stream of commerce flowing in a double tide inward and outward, while the footpaths were black with the hurrying swarm of pedestrians. It was difficult to realise as we looked at the line of fine shops and stately business premises that they really abutted on the other side upon the faded and stagnant square which we had just quitted. Sherlock Holmes was not very communicative during the long drive and lay back in the cab humming the tunes which he had heard in the afternoon. We rattled through an endless labyrinth of gas-lit streets until we emerged into Farrington Street. We had reached the same crowded thoroughfare in which we had found ourselves in the morning. Our cabs were dismissed, and, following the guidance of Mr. Merryweather, we passed down a narrow passage and through a side door, which he opened for us. Within there was a small corridor, which ended in a very massive iron gate. This also was opened, and led down a flight of winding stone steps, which terminated at another formidable gate. Mr. Merryweather stopped to light a lantern, and then conducted us down a dark, earth-smelling passage, and so, after opening a third door, into a huge vault or cellar, which was piled all round with crates and massive boxes. It was nearly four o'clock when we at last, after passing through the beautiful Stroud Valley, and over the broad gleaming Severn, found ourselves at the pretty little country-town of Ross. We had the carriage to ourselves save for an immense litter of papers which Holmes had brought with him. Among these he rummaged and read, with intervals of note-taking and of meditation, until we were past Reading. Then he suddenly rolled them all into a gigantic ball and tossed them up onto the rack. While Sherlock Holmes had been detailing this singular series of events, we had been whirling through the outskirts of the great town until the last straggling houses had been left behind, and we rattled along with a country hedge upon either side of us. Just as he finished, however, we drove through two scattered villages, where a few lights still glimmered in the windows. We made our way downstairs as quietly as possible, and out into the bright morning sunshine. We both sprang in, and away we dashed down the London Road. A few country carts were stirring, bearing in vegetables to the metropolis, but the lines of villas on either side were as silent and lifeless as some city in a dream. In town the earliest risers were just beginning to look sleepily from their windows as we drove through the streets of the Surrey side. Passing down the Waterloo Bridge Road we crossed over the river, and dashing up Wellington Street wheeled sharply to the right and found ourselves in Bow Street. My friend rose lazily from his armchair and stood with his hands in the pockets of his dressing-gown, looking over my shoulder. It was a bright, crisp February morning, and the snow of the day before still lay deep upon the ground, shimmering brightly in the wintry sun. Down the centre of Baker Street it had been ploughed into a brown crumbly band by the traffic, but at either side and on the heaped-up edges of the foot-paths it still lay as white as when it fell. The grey pavement had been cleaned and scraped, but was still dangerously slippery, so that there were fewer passengers than usual. It was a cold morning of the early spring, and we sat after breakfast on either side of a cheery fire in the old room at Baker Street. A thick fog rolled down between the lines of dun-coloured houses, and the opposing windows loomed like dark, shapeless blurs through the heavy yellow wreaths. Our gas was lit and shone on the white cloth and glimmer of china and metal, for the table had not been cleared yet. Sherlock Holmes had been silent all the morning, dipping continuously into the advertisement columns of a succession of papers until at last, having apparently given up his search, he had emerged in no very sweet temper to lecture me upon my literary shortcomings. By eleven o'clock the next day we were well upon our way to the old English capital. Holmes had been buried in the morning papers all the way down, but after we had passed the Hampshire border he threw them down and began to admire the scenery. It was an ideal spring day, a light blue sky, flecked with little fleecy white clouds drifting across from west to east. The sun was shining very brightly, and yet there was an exhilarating nip in the air, which set an edge to a man's energy. All over the countryside, away to the rolling hills around Aldershot, the little red and grey roofs of the farm-steadings peeped out from amid the light green of the new foliage. We were as good as our word, for it was just seven when we reached the Copper Beeches, having put up our trap at a wayside public-house. The group of trees, with their dark leaves shining like burnished metal in the light of the setting sun, were sufficient to mark the house even had Miss Hunter not been standing smiling on the door-step. About nine o'clock the light among the trees was extinguished, and all was dark in the direction of the Manor House. Two hours passed slowly away, and then, suddenly, just at the stroke of eleven, a single bright light shone out right in front of us. I had been delayed at a case, and it was a little after half-past six when I found myself in Baker Street once more. As I approached the house I saw a tall man in a Scotch bonnet with a coat which was buttoned up to his chin waiting outside in the bright semicircle which was thrown from the fanlight. Just as I arrived the door was opened, and we were shown up together to Holmes' room. I waited until midnight, but there was no sign of his return, so I retired to my room. It was no uncommon thing for him to be away for days and nights on end when he was hot upon a scent, so that his lateness caused me no surprise. I do not know at what hour he came in, but when I came down to breakfast in the morning there he was with a cup of coffee in one hand and the paper in the other, as fresh and trim as possible. It was not until close upon six o'clock that I found myself free and was able to spring into a hansom and drive to Baker Street, half afraid that I might be too late to assist at the denouement of the little mystery. I found Sherlock Holmes alone, however, half asleep, with his long, thin form curled up in the recesses of his armchair. A formidable array of bottles and test-tubes, with the pungent cleanly smell of hydrochloric acid, told me that he had spent his day in the chemical work which was so dear to him. One morning, at a little before seven o'clock, I was awakened by the maid tapping at the door to announce that two men had come from Paddington and were waiting in the consulting-room. I dressed hurriedly, for I knew by experience that railway cases were seldom trivial, and hastened downstairs. As I descended, my old ally, the guard, came out of the room and closed the door tightly behind him. "I never hope to see such a sight as that again, Mr. Holmes. From north, south, east, and west every man who had a shade of red in his hair had tramped into the city to answer the advertisement. Fleet Street was choked with red-headed folk, and Pope's Court looked like a coster's orange barrow. I should not have thought there were so many in the whole country as were brought together by that single advertisement. When I saw how many were waiting, I would have given it up in despair; but Spaulding would not hear of it. How he did it I could not imagine, but he pushed and pulled and butted until he got me through the crowd, and right up to the steps which led to the office. There was a double stream upon the stair, some going up in hope, and some coming back dejected; but we wedged in as well as we could and soon found ourselves in the office." "There was nothing in the office but a couple of wooden chairs and a deal table, behind which sat a small man with a head that was even redder than mine. He said a few words to each candidate as he came up, and then he always managed to find some fault in them which would disqualify them. Getting a vacancy did not seem to be such a very easy matter, after all." "However, in the morning I determined to have a look at it anyhow, so I bought a penny bottle of ink, and with a quill-pen, and seven sheets of foolscap paper, I started off for Pope's Court. "Well, to my surprise and delight, everything was as right as possible. The table was set out ready for me, and Mr. Duncan Ross was there to see that I got fairly to work. He started me off upon the letter A, and then he left me; but he would drop in from time to time to see that all was right with me. At two o'clock he bade me good-day, complimented me upon the amount that I had written, and locked the door of the office after me. "This went on day after day, Mr. Holmes, and on Saturday the manager came in and planked down four golden sovereigns for my week's work. It was the same next week, and the same the week after. Every morning I was there at ten, and every afternoon I left at two. By degrees Mr. Duncan Ross took to coming in only once of a morning, and then, after a time, he did not come in at all. Still, of course, I never dared to leave the room for an instant, for I was not sure when he might come, and the billet was such a good one, and suited me so well, that I would not risk the loss of it. "Eight weeks passed away like this, and I had written about Abbots and Archery and Armour and Architecture and Attica, and hoped with diligence that I might get on to the B's before very long. It cost me something in foolscap, and I had pretty nearly filled a shelf with my writings."

`scripts/01_make_dataset.py` - sharp and blurred images

In [ ]:
%%writefile scripts/01_make_dataset.py
"""Step 1: sharp text images and their blurred versions for every condition.

Conditions:
  blur model : sinc   - exactly the 1D filter's model
               box    - same horizontal blur done in pixels (mean of L neighbours, mirrored edges)
               angle8 - straight motion at 8 degrees
               shake  - hand shake: curved path + phone-like processing
  L          : blur length, px: 10, 20, 32 (beta = pi * L / n)
  sigma      : noise std, grey levels: 0, 2, 5, 10
4 x 3 x 4 = 48 conditions for every sharp image.

Every test image goes through all 48 conditions (paired design). Validation images are separate
(text continues further in the corpus) and are only used to pick alpha^2.

Output in --out:
  sharp/val_000.png ... sharp/test_029.png  - 5 + 30 sharp images 512x256
  degraded/<condition>__<id>.png            - 35 x 48 = 1680 blurred (1440 of them test)
  meta.csv                                  - file, sharp id, split, model, L, sigma, true beta
  gt_text.json                              - ground-truth text of every sharp image

  python scripts/01_make_dataset.py --out data --n-test 30 --n-val 5 --corpus texts/holmes_scenes.txt
"""
import argparse
import json
import os
import sys
import zlib

import numpy as np
import pandas as pd
from PIL import Image

sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))
from deblur.degradation import degrade          # noqa: E402
from deblur.textgen import Corpus, make_text_image, find_fonts  # noqa: E402


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="data")
    ap.add_argument("--n-test", type=int, default=10)
    ap.add_argument("--n-val", type=int, default=3)
    ap.add_argument("--width", type=int, default=512)
    ap.add_argument("--height", type=int, default=256)
    ap.add_argument("--models", default="sinc,box,angle8,shake")
    ap.add_argument("--lengths", default="10,20,32")
    ap.add_argument("--sigmas", default="0,2,5,10")
    ap.add_argument("--lang", default="eng", choices=["eng", "ukr"])
    ap.add_argument("--seed", type=int, default=2026)
    ap.add_argument("--corpus", default="", help="text file; empty = random words")
    a = ap.parse_args()

    # fixed seed: same images every run (the notebook checks an md5)
    rng = np.random.default_rng(a.seed)
    fonts = find_fonts()
    print("fonts:", fonts or "Pillow default")
    os.makedirs(os.path.join(a.out, "sharp"), exist_ok=True)
    os.makedirs(os.path.join(a.out, "degraded"), exist_ok=True)

    # 1. sharp images: validation first, then test; text runs on, so no overlap
    corpus = Corpus(a.corpus) if a.corpus else None
    gt, sharp_rows = {}, []
    for split, n in (("val", a.n_val), ("test", a.n_test)):
        for i in range(n):
            sid = f"{split}_{i:03d}"
            img, text = make_text_image(rng, a.width, a.height, a.lang, fonts, corpus)
            Image.fromarray(img).save(os.path.join(a.out, "sharp", sid + ".png"))
            gt[sid] = text
            sharp_rows.append((sid, split))

    # 2. blurred versions: every sharp image x 4 models x 3 lengths x 4 noise levels
    models = a.models.split(",")
    lengths = [float(x) for x in a.lengths.split(",")]
    sigmas = [float(x) for x in a.sigmas.split(",")]
    rows = []
    for sid, split in sharp_rows:
        img = np.array(Image.open(os.path.join(a.out, "sharp", sid + ".png")))
        for m in models:
            for L in lengths:
                for s in sigmas:
                    cond = f"{m}_L{int(L)}_s{int(s)}"          # e.g. "angle8_L20_s2"
                    # own seed per (image, condition), so noise and shake path don't depend on order
                    g, beta = degrade(img, m, L, s, np.random.default_rng(zlib.crc32(f"{a.seed}|{sid}|{cond}".encode())))
                    fn = f"{cond}__{sid}.png"
                    Image.fromarray(g).save(os.path.join(a.out, "degraded", fn))
                    rows.append(dict(file=fn, sharp_id=sid, split=split, blur_model=m, length_px=L,
                                     sigma=s, beta_true=beta, condition=cond))
    # 3. table of blurred images and the true texts
    pd.DataFrame(rows).to_csv(os.path.join(a.out, "meta.csv"), index=False)
    with open(os.path.join(a.out, "gt_text.json"), "w", encoding="utf-8") as f:
        json.dump(gt, f, ensure_ascii=False, indent=1)
    print(f"done: {len(sharp_rows)} sharp, {len(rows)} blurred images in {a.out}")


if __name__ == "__main__":
    main()

`scripts/02_run_experiment.py` - main experiment

In [ ]:
%%writefile scripts/02_run_experiment.py
"""Step 2: main experiment, quality and cost (time, energy) of every method.

Methods (names as in the result files):
  blurred        - no restoration (baseline)
  pinv_oracle    - 1D pseudo-inverse filter with the true beta (upper bound for the filter)
  pinv_blind     - 1D filter, beta from the cepstrum ("Pseudo-inverse 1D")
  pinv2d_blind   - 2D filter, L and angle from the 2D cepstrum ("Pseudo-inverse 2D")
  dgv2_mobilenet - DeblurGAN-v2, MobileNet-v2 backbone (~3.3 M params)
  dgv2_inception - DeblurGAN-v2, Inception-ResNet-v2 backbone (~61 M params)

alpha^2 is not tuned on test data: it is chosen per noise level on validation images,
and on test the noise level is estimated automatically (skimage estimate_sigma).

Output in --out:
  quality.csv           - one row per (test image, method): PSNR, SSIM, CER, WER, cer_sharp, estimated beta / L / angle
  efficiency.csv        - time (ms per image) and energy (J per image), CPU and GPU models
  alpha_selection.json  - chosen alpha^2 per noise level (1D and 2D)
  alpha_sensitivity.csv - validation PSNR for every alpha^2
  summary_*.csv         - mean metrics by method and condition
  stats_wilcoxon.csv    - first comparison per image (the final one, per text, is in 04_stats.py)
  restored/<method>/... - all restored images (too big for the zip)

  python scripts/02_run_experiment.py --data data --out results --energy-seconds 30
"""
import argparse
import json
import os
import sys
import time

import numpy as np
import pandas as pd
from PIL import Image
from scipy.stats import wilcoxon
from skimage.restoration import estimate_sigma

sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))
from deblur.classical import ALPHA2_GRID, estimate_beta, pseudo_inverse  # noqa: E402
from deblur.classical2d import estimate_motion_2d, deconv2d  # noqa: E402
from deblur import metrics as M  # noqa: E402
from deblur.energy import energy_per_item, measure_idle_power, time_per_item  # noqa: E402


def load(path):
    return np.array(Image.open(path), dtype=np.float64)


def true_angle(blur_model):
    # 8 for "angle8", 0 otherwise (shake has no angle)
    return float(blur_model[len("angle"):] or 8) if blur_model.startswith("angle") else 0.0


def select_alpha(meta, data_dir):
    """For each noise level, the alpha^2 with the best mean validation PSNR.
    Separately for the 1D filter (true beta) and the 2D filter (true L and angle).
    Blur parameters are known here, so only the effect of alpha^2 is measured."""
    val = meta[meta.split == "val"]
    rows = []
    for _, r in val.iterrows():
        g = load(os.path.join(data_dir, "degraded", r.file))
        f = load(os.path.join(data_dir, "sharp", r.sharp_id + ".png"))
        # keep the noise estimate, used later to map test images to a noise level
        sh = float(estimate_sigma(g))
        for a2 in ALPHA2_GRID:
            rows.append(dict(filter="1d", sigma=r.sigma, blur_model=r.blur_model, alpha2=a2,
                             psnr=M.psnr(f, pseudo_inverse(g, r.beta_true, a2)), sigma_hat=sh))
            rows.append(dict(filter="2d", sigma=r.sigma, blur_model=r.blur_model, alpha2=a2,
                             psnr=M.psnr(f, deconv2d(g, r.length_px, true_angle(r.blur_model), a2)),
                             sigma_hat=sh))
    sens = pd.DataFrame(rows)
    lookups = {}
    for flt, d in sens.groupby("filter"):
        table = d.groupby(["sigma", "alpha2"]).psnr.mean().reset_index()
        best = table.loc[table.groupby("sigma").psnr.idxmax()][["sigma", "alpha2"]]
        lookups[flt] = dict(zip(best.sigma, best.alpha2))
    # our run: 1D {0: 1e-4, 2: 1e-3, 5: 1e-2, 10: 1e-2}, 2D {0: 1e-3, 2: 1e-3, 5: 1e-2, 10: 1e-2}
    # mean noise estimate for each true level, to map sigma_hat -> level
    sig_hat = sens.groupby("sigma").sigma_hat.mean().to_dict()
    return lookups, sig_hat, sens


def make_alpha_picker(lookup, sig_hat):
    # pick(g): estimate the noise (Donoho: median of the finest wavelet coefficients / 0.6745),
    # take the closest validation level and return its alpha^2. No ground truth is used.
    levels = sorted(lookup)
    centers = np.array([sig_hat[s] for s in levels])

    def pick(g):
        s = float(estimate_sigma(g))
        return lookup[levels[int(np.argmin(np.abs(centers - s)))]]
    return pick


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data")
    ap.add_argument("--out", default="results")
    ap.add_argument("--dg-repo", default="DeblurGANv2")
    ap.add_argument("--w-mobilenet", default="weights/fpn_mobilenet.h5")
    ap.add_argument("--w-inception", default="weights/fpn_inception.h5")
    ap.add_argument("--lang", default="eng")
    ap.add_argument("--no-ocr", action="store_true")
    ap.add_argument("--eff-condition", default="sinc_L32_s2",
                    help="condition used for time and energy")
    ap.add_argument("--energy-seconds", type=float, default=30.0)
    ap.add_argument("--save-restored", action="store_true", default=True)
    ap.add_argument("--only", default="", help="run only these methods (comma separated) and add them to existing results")
    a = ap.parse_args()

    os.makedirs(a.out, exist_ok=True)
    meta = pd.read_csv(os.path.join(a.data, "meta.csv"))
    gt = json.load(open(os.path.join(a.data, "gt_text.json"), encoding="utf-8"))

    # 1. alpha^2 on validation
    print("1) picking alpha^2 on validation images")
    lookups, sig_hat, sens = select_alpha(meta, a.data)
    sens.to_csv(os.path.join(a.out, "alpha_sensitivity.csv"), index=False)
    json.dump({"alpha2_by_sigma_1d": {str(k): v for k, v in lookups["1d"].items()},
               "alpha2_by_sigma_2d": {str(k): v for k, v in lookups["2d"].items()},
               "sigma_hat_by_sigma": {str(k): v for k, v in sig_hat.items()}},
              open(os.path.join(a.out, "alpha_selection.json"), "w"), indent=1)
    print("   alpha^2 by noise level, 1D:", lookups["1d"], " 2D:", lookups["2d"])
    pick_alpha = make_alpha_picker(lookups["1d"], sig_hat)
    pick_alpha2d = make_alpha_picker(lookups["2d"], sig_hat)

    # 2. methods: name -> function(blurred) -> restored
    methods = {
        "blurred": lambda g: g,
        "pinv_oracle": None,  # needs beta_true, see below
        "pinv_blind": lambda g: pseudo_inverse(g, estimate_beta(g), pick_alpha(g)),
        "pinv2d_blind": None,  # see below
    }
    # both DeblurGAN-v2 nets, on GPU if there is one
    nets = {}
    try:
        import torch
        from deblur.deblurgan import DeblurGANv2
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        for name, bb, w in (("dgv2_mobilenet", "fpn_mobilenet", a.w_mobilenet),
                            ("dgv2_inception", "fpn_inception", a.w_inception)):
            if w and os.path.exists(w):
                nets[name] = DeblurGANv2(a.dg_repo, w, bb, device=dev)
                methods[name] = nets[name]
                print(f"   {name}: {nets[name].n_params / 1e6:.2f} M params, {dev}")
            else:
                print(f"   {name}: no weights at {w}, skipped")
    except Exception as e:  # noqa: BLE001
        print("   DeblurGAN-v2 not available:", e)

    # --only reruns some methods (not used in our run)
    only = [m for m in a.only.split(",") if m]
    if only:
        methods = {k: v for k, v in methods.items() if k in only}
        nets = {k: v for k, v in nets.items() if k in only}
        print("   only:", list(methods))

    # 3. quality on all 1440 test images, every method
    print("2) restoring test images")
    test = meta[meta.split == "test"].reset_index(drop=True)
    rows = []
    sharp_ocr = {}
    t0 = time.time()
    for i, r in test.iterrows():
        g = load(os.path.join(a.data, "degraded", r.file))
        f = load(os.path.join(a.data, "sharp", r.sharp_id + ".png"))
        # OCR on the sharp image once per text: OCR itself should make almost no errors
        if not a.no_ocr and r.sharp_id not in sharp_ocr:
            sharp_ocr[r.sharp_id] = M.cer(gt[r.sharp_id], M.ocr(f, a.lang))
        for name, fn in methods.items():
            extra = {}
            if name == "pinv_oracle":
                # true beta (we blurred it ourselves)
                out = pseudo_inverse(g, r.beta_true, pick_alpha(g))
            elif name == "pinv_blind":
                # beta from the cepstrum, saved to check the error later
                b = estimate_beta(g)
                extra["beta_est"] = b
                out = pseudo_inverse(g, b, pick_alpha(g))
            elif name == "pinv2d_blind":
                L2, th2 = estimate_motion_2d(g)
                extra.update(len2d_est=L2, angle2d_est=th2)
                out = deconv2d(g, L2, th2, pick_alpha2d(g))
            else:
                out = fn(g)                                       # blurred or a network
            row = dict(file=r.file, sharp_id=r.sharp_id, condition=r.condition, blur_model=r.blur_model,
                       length_px=r.length_px, sigma=r.sigma, beta_true=r.beta_true, method=name,
                       psnr=M.psnr(f, out), ssim=M.ssim(f, out), **extra)
            if not a.no_ocr:
                hyp = M.ocr(out, a.lang)
                row.update(cer=M.cer(gt[r.sharp_id], hyp), wer=M.wer(gt[r.sharp_id], hyp),
                           cer_sharp=sharp_ocr[r.sharp_id])
            rows.append(row)
            if a.save_restored:
                d = os.path.join(a.out, "restored", name)
                os.makedirs(d, exist_ok=True)
                Image.fromarray(np.clip(np.round(out), 0, 255).astype(np.uint8)).save(os.path.join(d, r.file))
        if (i + 1) % 20 == 0:
            print(f"   {i + 1}/{len(test)} images, {time.time() - t0:.0f} s")
    q = pd.DataFrame(rows)
    qpath = os.path.join(a.out, "quality.csv")
    if only and os.path.exists(qpath):
        old = pd.read_csv(qpath)
        q = pd.concat([old[~old.method.isin(only)], q], ignore_index=True)
    q.to_csv(qpath, index=False)

    # 4. time and energy on the 30 images of one condition (sinc, L = 32, sigma = 2);
    # both the filter and the nets do the same work for any blur on a 512x256 image
    print("3) time and energy")
    sub = test[test.condition == a.eff_condition]
    items = [(load(os.path.join(a.data, "degraded", r.file)), r.beta_true) for _, r in sub.iterrows()]
    idle = measure_idle_power(min(20.0, a.energy_seconds))
    print(f"   idle power: {idle:.1f} W")
    eff = []

    def bench(name, fn, device, n_params):
        # time per image -> mean ms; energy over >= 30 s loop -> J per image
        # for the filters the time includes noise estimate, cepstrum and filtering
        ts = time_per_item(fn, items)
        en = energy_per_item(fn, items, a.energy_seconds, idle, project=name)
        eff.append(dict(method=name, device=device, n_params=n_params,
                        ms_per_img_mean=1000 * ts.mean(), ms_per_img_std=1000 * ts.std(),
                        idle_power_W=idle, **en))
        print(f"   {name:15s} {device:4s} {1000 * ts.mean():9.1f} ms/img  "
              f"{en['net_J_per_img']:.3f} J/img net  [cpu: {en['cpu_tracking']}]")

    def deconv2d_blind(g):
        L2, th2 = estimate_motion_2d(g)
        return deconv2d(g, L2, th2, pick_alpha2d(g))

    # filters on CPU only
    if "pinv_oracle" in methods:
        bench("pinv_oracle", lambda it: pseudo_inverse(it[0], it[1], pick_alpha(it[0])), "cpu", 0)
    if "pinv_blind" in methods:
        bench("pinv_blind", lambda it: pseudo_inverse(it[0], estimate_beta(it[0]), pick_alpha(it[0])), "cpu", 0)
    if "pinv2d_blind" in methods:
        bench("pinv2d_blind", lambda it: deconv2d_blind(it[0]), "cpu", 0)
    # networks on CPU (separate copy) and on GPU
    for name, net in nets.items():
        from deblur.deblurgan import DeblurGANv2
        bb = net.backbone
        w = a.w_mobilenet if bb == "fpn_mobilenet" else a.w_inception
        cpu_net = net if net.device == "cpu" else DeblurGANv2(a.dg_repo, w, bb, device="cpu")
        bench(name, lambda it, n=cpu_net: n(it[0]), "cpu", net.n_params)
        if net.device.startswith("cuda"):
            bench(name, lambda it, n=net: n(it[0]), "cuda", net.n_params)
    eff = pd.DataFrame(eff)
    epath = os.path.join(a.out, "efficiency.csv")
    if only and os.path.exists(epath):
        old = pd.read_csv(epath)
        eff = pd.concat([old[~old.method.isin(only)], eff], ignore_index=True)
    eff.to_csv(epath, index=False)

    # 5. summary tables
    print("4) summary tables")
    agg_cols = [c for c in ("psnr", "ssim", "cer", "wer") if c in q]
    overall = q.groupby("method")[agg_cols].agg(["mean", "std"]).round(4)
    overall.to_csv(os.path.join(a.out, "summary_overall.csv"))
    for key in ("blur_model", "sigma", "length_px", "condition"):
        q.groupby(["method", key])[agg_cols].mean().round(4).to_csv(
            os.path.join(a.out, f"summary_by_{key}.csv"))
    # first Wilcoxon test per image; images of one text are not independent,
    # so the real comparison is per text in 04_stats.py
    stats = []
    for bname in [m for m in ("pinv_blind", "pinv2d_blind") if m in q.method.unique()]:
        base = q[q.method == bname].set_index("file")
        for other in [m for m in q.method.unique() if m.startswith("dgv2")]:
            o = q[q.method == other].set_index("file")
            for bm in ["all"] + sorted(q.blur_model.unique()):
                idx = base.index if bm == "all" else base[base.blur_model == bm].index
                for col in agg_cols:
                    x, y = base.loc[idx, col], o.loc[idx, col]
                    try:
                        p = wilcoxon(x, y).pvalue
                    except ValueError:
                        p = float("nan")
                    stats.append(dict(compare=f"{bname} vs {other}", blur_model=bm, metric=col,
                                      mean_classical=x.mean(), mean_other=y.mean(), p_wilcoxon=p, n=len(idx)))
    pd.DataFrame(stats).to_csv(os.path.join(a.out, "stats_wilcoxon.csv"), index=False)
    if "beta_est" in q:
        b = q[q.method == "pinv_blind"]
        err = (b.beta_est - b.beta_true).abs()
        print(f"   beta error: median {err.median():.4f}, 95th percentile {err.quantile(.95):.4f}")
    print(overall)
    print(f"done, results in {a.out}")


if __name__ == "__main__":
    main()

`scripts/03_make_figures.py` - extra figures and the summary table

In [ ]:
%%writefile scripts/03_make_figures.py
"""Step 3: extra figures and the summary table, to look through the results.

Quality vs blur length and noise, quality vs energy, restoration examples,
validation PSNR vs alpha^2. Table goes to results/table_main.csv (.md).

  python scripts/03_make_figures.py --data data --res results
"""
import argparse
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from PIL import Image  # noqa: E402

# labels
SHORT = {"blurred": "Blurred", "pinv_oracle": "Pseudo-inverse\n(known β)", "pinv_blind": "Pseudo-inverse 1D\n(estimated)",
         "pinv2d_blind": "Pseudo-inverse 2D\n(estimated)",
         "dgv2_mobilenet": "DeblurGAN-v2\nMobileNet", "dgv2_inception": "DeblurGAN-v2\nInception"}
LABELS = {"blurred": "Blurred", "pinv_oracle": "Pseudo-inverse (known β)",
          "pinv_blind": "Pseudo-inverse 1D (estimated β)", "pinv2d_blind": "Pseudo-inverse 2D (estimated L, θ)",
          "dgv2_mobilenet": "DeblurGAN-v2 MobileNet", "dgv2_inception": "DeblurGAN-v2 Inception"}
STYLE = {"blurred": ("0.65", ":", "x"), "pinv_oracle": ("0.0", "--", "s"),
         "pinv_blind": ("0.0", "-", "o"), "pinv2d_blind": ("0.0", "-", "v"), "dgv2_mobilenet": ("0.45", "-.", "^"),
         "dgv2_inception": ("0.3", "-", "D")}
MODEL_UA = {"sinc": "sinc model", "box": "box kernel", "angle8": "motion at 8°",
            "shake": "hand shake + JPEG"}

plt.rcParams.update({"font.family": "serif", "font.size": 9, "axes.grid": True, "grid.color": "0.85"})


def save(fig, out, name):
    if not fig.get_constrained_layout():
        fig.tight_layout()
    fig.savefig(os.path.join(out, name + ".jpg"), dpi=300, pil_kwargs={"quality": 95})
    fig.savefig(os.path.join(out, name + ".png"), dpi=200)
    plt.close(fig)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data")
    ap.add_argument("--res", default="results")
    ap.add_argument("--example-file", default=None, help="blurred image file for the examples figure")
    a = ap.parse_args()
    out = os.path.join(a.res, "figures")
    os.makedirs(out, exist_ok=True)
    q = pd.read_csv(os.path.join(a.res, "quality.csv"))
    methods = [m for m in LABELS if m in q.method.unique()]
    metric_ocr = "cer" in q

    # quality vs blur length, one panel per blur model
    for metric, ylab in (("psnr", "PSNR, dB"), ("cer", "Character error rate")):
        if metric not in q:
            continue
        bms = sorted(q.blur_model.unique(), key=lambda s: list(MODEL_UA).index(s) if s in MODEL_UA else 9)
        fig, axes = plt.subplots(1, len(bms), figsize=(2.6 * len(bms), 2.5), sharey=True)
        axes = np.atleast_1d(axes)
        for ax, bm in zip(axes, bms):
            d = q[q.blur_model == bm].groupby(["method", "length_px"])[metric].mean().reset_index()
            for m in methods:
                dm = d[d.method == m]
                c, ls, mk = STYLE[m]
                ax.plot(dm.length_px, dm[metric], color=c, ls=ls, marker=mk, ms=4, label=LABELS[m])
            ax.set_title(MODEL_UA.get(bm, bm))
            ax.set_xlabel("Blur length L, px")
        axes[0].set_ylabel(ylab)
        axes[-1].legend(fontsize=6.5, loc="best")
        save(fig, out, f"fig1_{metric}_vs_length")

    # quality vs noise
    metric = "cer" if metric_ocr else "psnr"
    d = q.groupby(["method", "sigma"])[metric].mean().reset_index()
    fig, ax = plt.subplots(figsize=(3.4, 2.5))
    for m in methods:
        dm = d[d.method == m]
        c, ls, mk = STYLE[m]
        ax.plot(dm.sigma, dm[metric], color=c, ls=ls, marker=mk, ms=4, label=LABELS[m])
    ax.set_xlabel("Noise σ, grey levels")
    ax.set_ylabel("CER" if metric_ocr else "PSNR, dB")
    ax.legend(fontsize=6.5)
    save(fig, out, f"fig2_{metric}_vs_noise")

    # quality vs energy
    eff_path = os.path.join(a.res, "efficiency.csv")
    if os.path.exists(eff_path):
        eff = pd.read_csv(eff_path)
        eff_cpu = eff[eff.device == "cpu"].set_index("method")
        cond = eff.get("condition", None)
        mq = q.groupby("method")[[c for c in ("psnr", "cer") if c in q]].mean()
        fig, ax = plt.subplots(figsize=(3.4, 2.6))
        yk = "cer" if metric_ocr else "psnr"
        for m in methods:
            if m in eff_cpu.index:
                c, _, mk = STYLE[m]
                x = max(eff_cpu.loc[m, "gross_J_per_img"], 1e-4)
                ax.scatter(x, mq.loc[m, yk], color=c, marker=mk, s=40, label=LABELS[m], zorder=3)
        ax.set_xscale("log")
        ax.set_xlabel("Energy per image (CPU), J")
        ax.set_ylabel("Mean CER" if yk == "cer" else "Mean PSNR, dB")
        ax.legend(fontsize=6.5)
        save(fig, out, "fig3_quality_vs_energy")
        del cond

    # restoration examples
    test = q[q.method == "blurred"]
    ex_files = [a.example_file] if a.example_file else []
    if not ex_files:
        for cond in ("sinc_L32_s2", "angle8_L20_s5", "shake_L20_s2"):
            f = test[test.condition == cond].file
            if len(f):
                ex_files.append(f.iloc[0])
        if not ex_files:   # other conditions: take any two
            ex_files = list(test.drop_duplicates("condition").file.iloc[[0, -1]])
    cols = ["sharp"] + methods
    fig, axes = plt.subplots(len(ex_files), len(cols), figsize=(1.25 * len(cols), 1.3 * len(ex_files) + 0.3),
                             gridspec_kw={"wspace": 0.05, "hspace": 0.15})
    axes = np.atleast_2d(axes)
    for i, fn in enumerate(ex_files):
        sid = fn.split("__")[1].replace(".png", "")
        for j, c in enumerate(cols):
            p = os.path.join(a.data, "sharp", sid + ".png") if c == "sharp" else os.path.join(a.res, "restored", c, fn)
            img = np.array(Image.open(p))
            ax = axes[i, j]
            ax.imshow(img[:, : img.shape[1] // 2], cmap="gray", vmin=0, vmax=255)
            ax.set_xticks([]), ax.set_yticks([])
            if i == 0:
                ax.set_title("Sharp" if c == "sharp" else SHORT.get(c, LABELS[c]), fontsize=6.5)
        axes[i, 0].set_ylabel(fn.split("__")[0], fontsize=6)
    save(fig, out, "fig4_examples")

    # validation PSNR vs alpha^2
    sp = os.path.join(a.res, "alpha_sensitivity.csv")
    if os.path.exists(sp):
        s = pd.read_csv(sp)
        if "filter" in s:                      # 1D filter only
            s = s[s["filter"] == "1d"]
        s = s.groupby(["sigma", "alpha2"]).psnr.mean().reset_index()
        fig, ax = plt.subplots(figsize=(3.4, 2.5))
        for k, (sig, d) in enumerate(s.groupby("sigma")):
            ax.plot(d.alpha2, d.psnr, color=str(0.15 * k), marker="os^Dv"[k % 5], ms=3.5, label=f"σ = {sig:g}")
        ax.set_xscale("log")
        ax.set_xlabel("α²")
        ax.set_ylabel("PSNR, dB (validation)")
        ax.legend(fontsize=6.5)
        save(fig, out, "fig5_alpha_sensitivity")

    # summary table
    rows = []
    eff = pd.read_csv(eff_path) if os.path.exists(eff_path) else pd.DataFrame()
    for m in methods:
        d = q[q.method == m]
        r = {"Method": LABELS[m], "PSNR, dB": f"{d.psnr.mean():.2f}", "SSIM": f"{d.ssim.mean():.3f}"}
        if metric_ocr:
            r["CER"] = f"{d.cer.mean():.3f}"
        for dev in ("cpu", "cuda"):
            e = eff[(eff.method == m) & (eff.device == dev)] if len(eff) else eff
            if len(e):
                r[f"Time {dev.upper()}, ms"] = f"{e.ms_per_img_mean.iloc[0]:.1f}"
                r[f"Energy {dev.upper()}, J"] = f"{e.gross_J_per_img.iloc[0]:.3f}"
        if len(eff) and (eff.method == m).any():
            r["Params, M"] = f"{eff[eff.method == m].n_params.iloc[0] / 1e6:.1f}"
        rows.append(r)
    t = pd.DataFrame(rows).fillna("–")
    t.to_csv(os.path.join(a.res, "table_main.csv"), index=False)
    try:
        open(os.path.join(a.res, "table_main.md"), "w", encoding="utf-8").write(t.to_markdown(index=False))
    except ImportError:
        pass
    print(t.to_string(index=False))
    print("figures:", out)


if __name__ == "__main__":
    main()

`scripts/04_stats.py` - statistics per text

In [ ]:
%%writefile scripts/04_stats.py
"""Step 4: statistics, means with 95 % CIs and paired comparisons of the methods.

Each of the 30 test texts is blurred under 48 conditions, so the 1440 images are not
1440 independent samples (a text that is hard for OCR is hard under every condition).
So the unit is the source text (sharp_id), n = 30:
  * CER of each method is averaged over all conditions of one text (per blur model and overall)
  * CI by bootstrap over texts (5000 resamples)
  * methods compared with the paired Wilcoxon test over texts

Output in results/:
  stats_ci.csv            mean and 95 % CI per method, metric and blur model
  stats_paired.csv        differences classical - network, 95 % CI, p (Wilcoxon),
                          share of texts where the classical one is better (share_a_better)
  stats_2d_estimation.csv median errors of L and angle for the 2D filter

  python scripts/04_stats.py --synthetic results
"""
import argparse
import os

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

# bootstrap resamples and a fixed seed
B = 5000
RNG = np.random.default_rng(2026)
CLASSICAL = ("pinv_blind", "pinv2d_blind", "pinv1d_blind", "pinv_oracle")


# bootstrap CI of the mean: resample with replacement, take the 2.5 and 97.5 percentiles
# of the means. Returns (mean, low, high).
def boot_ci(x, b=B):
    x = np.asarray(x, float)
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    idx = RNG.integers(0, len(x), (b, len(x)))
    m = x[idx].mean(1)
    return float(x.mean()), float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))


# paired comparison of methods a and b over units (texts)
# piv: row = text, column = method, value = mean metric of that text
# share_a_better: share of texts where a is better (lower for CER/WER)
def paired(piv, a, b, metric):
    d = piv[[a, b]].dropna()
    if len(d) < 2:
        return None
    diff = d[a] - d[b]
    mean, lo, hi = boot_ci(diff)
    try:
        p = float(wilcoxon(d[a], d[b]).pvalue)
    except ValueError:
        p = float("nan")
    better = (d[a] < d[b]).mean() if metric in ("cer", "wer") else (d[a] > d[b]).mean()
    return dict(compare=f"{a} - {b}", metric=metric, n_units=len(d), mean_a=float(d[a].mean()),
                mean_b=float(d[b].mean()), diff=mean, diff_lo=lo, diff_hi=hi, p_wilcoxon=p,
                share_a_better=float(better))


# unit = sharp_id (text), group = blur_model; "all" = all models together
def analyse(q, unit, group, out_prefix, metrics, exclude_from_all=()):
    ci_rows, pr_rows = [], []
    groups = ["all"] + sorted(q[group].dropna().unique()) if group else ["all"]
    for g in groups:
        if g == "all":
            sub = q[~q[group].isin(exclude_from_all)] if group else q
        else:
            sub = q[q[group] == g]
        for metric in metrics:
            if metric not in sub:
                continue
            # one number per text: mean over all conditions of the group
            # (e.g. angle8: 3 lengths x 4 noise levels = 12 images)
            per_unit = sub.groupby([unit, "method"])[metric].mean().unstack("method")
            for m in per_unit.columns:
                mean, lo, hi = boot_ci(per_unit[m])
                ci_rows.append(dict(group=g, metric=metric, method=m, n_units=int(per_unit[m].notna().sum()),
                                    mean=mean, ci_lo=lo, ci_hi=hi))
            nets = [m for m in per_unit.columns if m.startswith("dgv2")]
            cls = [m for m in per_unit.columns if m in CLASSICAL]
            # every classical filter vs every network, plus 2D vs 1D
            pairs = [(c, n) for c in cls for n in nets]
            if "pinv_blind" in cls and "pinv2d_blind" in cls:
                pairs.append(("pinv2d_blind", "pinv_blind"))
            if "pinv1d_blind" in cls and "pinv2d_blind" in cls:
                pairs.append(("pinv2d_blind", "pinv1d_blind"))
            for a, b in pairs:
                r = paired(per_unit, a, b, metric)
                if r:
                    pr_rows.append(dict(group=g, **r))
    ci, pr = pd.DataFrame(ci_rows), pd.DataFrame(pr_rows)
    ci.to_csv(out_prefix + "stats_ci.csv", index=False)
    pr.to_csv(out_prefix + "stats_paired.csv", index=False)
    with pd.option_context("display.width", 200, "display.max_rows", 500):
        print(ci[ci.metric == metrics[0]].round(4).to_string(index=False))
        print(pr[pr.metric == metrics[0]].round(4).to_string(index=False))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--synthetic", default="results")
    ap.add_argument("--real", nargs="*", default=[])
    a = ap.parse_args()

    # synthetic experiment
    qp = os.path.join(a.synthetic, "quality.csv")
    if os.path.exists(qp):
        q = pd.read_csv(qp)
        print("=== synthetic, unit = text, n =", q.sharp_id.nunique())
        analyse(q, "sharp_id", "blur_model", os.path.join(a.synthetic, ""), ["cer", "wer", "psnr", "ssim"])
        if "beta_est" in q:
            b = q[q.method == "pinv_blind"]
            # 1D length error in px: |beta_est - beta_true| * n / pi
            L_err = (b.beta_est - b.beta_true).abs() * 512 / np.pi
            print(f"L error (1D, px): median {L_err.median():.3f}, 95th percentile {L_err.quantile(.95):.3f}")
        if "len2d_est" in q:
            b = q[q.method == "pinv2d_blind"]
            # 2D errors: length in px, angle in degrees (period 180)
            # shake has no true angle (0 is used), so its angle error means nothing
            Lerr = (b.len2d_est - b.length_px).abs()
            ang_true = np.where(b.blur_model == "angle8", 8.0, 0.0)
            aerr = np.abs(((b.angle2d_est - ang_true) + 90) % 180 - 90)
            pd.DataFrame(dict(blur_model=b.blur_model, sigma=b.sigma, L_err=Lerr, angle_err=aerr)).groupby(
                "blur_model").agg(["median", lambda s: s.quantile(.95)]).round(3).to_csv(
                os.path.join(a.synthetic, "stats_2d_estimation.csv"))
            print(f"2D error: L median {Lerr.median():.3f} px, angle median {np.median(aerr):.2f}°")
    # real photos (optional, not used in the paper)
    for r in a.real:
        rp = os.path.join(r, "real_quality.csv")
        if not os.path.exists(rp):
            print("not found:", rp)
            continue
        q = pd.read_csv(rp)
        unit = "page" if "page" in q and q.page.notna().any() else "image"
        metric = "cer" if "cer" in q and q.cer.notna().any() else "ocr_conf"
        print(f"=== real photos {r}, unit = {unit}, n = {q[unit].nunique()}")
        analyse(q, unit, "category" if "category" in q else None,
                os.path.join(r, "real_"), [metric] + (["wer"] if metric == "cer" else []),
                exclude_from_all=("sharp",))   # "all" = blurred photos only


if __name__ == "__main__":
    main()

`scripts/05_atlas.py` - input → output for every condition, Fig. 2

In [ ]:
%%writefile scripts/05_atlas.py
"""Step 5: "atlas" of the experiment. For every blur condition: the input, the output of each
method and the parameters it used.

For the chosen test texts (test_000 and test_001 in the notebook) it goes through all 48 conditions
(4 motions x 3 lengths x 4 noise levels) and saves:
  * atlas/<condition>/<text>__<method>.png  - full size images (sharp, blurred, restored)
  * atlas_<text>.pdf                         - one page per condition with captions
                                               (blur parameters, estimated L and theta, alpha^2, CER, time)
  * atlas_params.csv                         - the same parameters as a table
  * fig_examples_en.jpg / fig_examples_uk.jpg - Fig. 2 of the paper, b/w, 300 dpi
alpha^2 is chosen the same way as in step 2 (validation texts, separately for 1D and 2D).
This is a rerun of the same methods on the same images (CER matches quality.csv),
only now the pictures and parameters are saved.

  python scripts/05_atlas.py --data data --out atlas_out --ids test_000,test_001
only redraw the figure from a finished atlas:  python scripts/05_atlas.py --fig-only --out atlas_out
"""
import argparse
import importlib.util
import json
import os
import sys
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
from matplotlib.backends.backend_pdf import PdfPages  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from PIL import Image  # noqa: E402

sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))
from deblur import metrics as M  # noqa: E402
from deblur.classical import estimate_beta, pseudo_inverse, beta_to_length  # noqa: E402
from deblur.classical2d import cepstrum2d, deconv2d, estimate_motion_2d  # noqa: E402

# reuse functions from 02_run_experiment.py (alpha^2 choice, loading) so the atlas does exactly
# the same thing; the file name starts with a digit, so it is loaded by path
_spec = importlib.util.spec_from_file_location("exp02", os.path.join(os.path.dirname(__file__), "02_run_experiment.py"))
exp02 = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(exp02)

MOTION = {"sinc": "straight horizontal (sinc model)", "box": "straight horizontal (spatial averaging)",
          "angle8": "straight at 8°", "shake": "hand shake + gamma + noise + JPEG"}
TITLE = {"sharp": "Sharp (reference)", "blurred": "Blurred input", "pinv_oracle": "Pseudo-inverse, known β",
         "pinv_blind": "Pseudo-inverse 1D (auto)", "pinv2d_blind": "Pseudo-inverse 2D (auto)",
         "dgv2_mobilenet": "DeblurGAN-v2 MobileNet", "dgv2_inception": "DeblurGAN-v2 Inception"}


def u8(x):
    return np.clip(np.round(x), 0, 255).astype(np.uint8)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data")
    ap.add_argument("--out", default="atlas_out")
    ap.add_argument("--ids", default="test_000", help="test texts, comma separated")
    ap.add_argument("--dg-repo", default="DeblurGANv2")
    ap.add_argument("--w-mobilenet", default="weights/fpn_mobilenet.h5")
    ap.add_argument("--w-inception", default="weights/fpn_inception.h5")
    ap.add_argument("--fig-only", action="store_true", help="only redraw the examples figure from atlas_params.csv")
    a = ap.parse_args()
    if a.fig_only:
        make_examples_figure(a.out, a.ids.split(",")[0])
        return
    os.makedirs(a.out, exist_ok=True)
    meta = pd.read_csv(os.path.join(a.data, "meta.csv"))
    gt = json.load(open(os.path.join(a.data, "gt_text.json"), encoding="utf-8"))

    print("alpha^2 on validation texts")
    lookups, sig_hat, _ = exp02.select_alpha(meta, a.data)
    pick1 = exp02.make_alpha_picker(lookups["1d"], sig_hat)
    pick2 = exp02.make_alpha_picker(lookups["2d"], sig_hat)

    nets = {}
    try:
        import torch
        from deblur.deblurgan import DeblurGANv2
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        for name, bb, w in (("dgv2_mobilenet", "fpn_mobilenet", a.w_mobilenet),
                            ("dgv2_inception", "fpn_inception", a.w_inception)):
            if os.path.exists(w):
                nets[name] = DeblurGANv2(a.dg_repo, w, bb, device=dev)
    except Exception as e:  # noqa: BLE001
        print("DeblurGAN-v2 not available:", e)
    print("networks:", list(nets) or "none")

    rows = []
    order = ["sinc", "box", "angle8", "shake"]
    # all 48 conditions of each text, order sinc -> box -> angle8 -> shake
    for sid in [s for s in a.ids.split(",") if s]:
        f = exp02.load(os.path.join(a.data, "sharp", sid + ".png"))
        sub = meta[(meta.sharp_id == sid)].copy()
        sub["o"] = sub.blur_model.map({m: i for i, m in enumerate(order)})
        sub = sub.sort_values(["o", "length_px", "sigma"])
        pdf = PdfPages(os.path.join(a.out, f"atlas_{sid}.pdf"))
        for _, r in sub.iterrows():
            g = exp02.load(os.path.join(a.data, "degraded", r.file))
            cond = r.condition
            d = os.path.join(a.out, "atlas", cond)
            os.makedirs(d, exist_ok=True)
            outs, info = {"sharp": f, "blurred": g}, {}
            # run every method, keep alpha^2, estimated L and theta, and time of one call
            a1, a2 = pick1(g), pick2(g)
            t0 = time.perf_counter(); outs["pinv_oracle"] = pseudo_inverse(g, r.beta_true, a1)
            info["pinv_oracle"] = dict(alpha2=a1, L=beta_to_length(r.beta_true, g.shape[1]), sec=time.perf_counter() - t0)
            t0 = time.perf_counter(); b = estimate_beta(g); outs["pinv_blind"] = pseudo_inverse(g, b, a1)
            info["pinv_blind"] = dict(alpha2=a1, L=beta_to_length(b, g.shape[1]), sec=time.perf_counter() - t0)
            t0 = time.perf_counter(); L2, th2 = estimate_motion_2d(g); outs["pinv2d_blind"] = deconv2d(g, L2, th2, a2)
            info["pinv2d_blind"] = dict(alpha2=a2, L=L2, theta=th2, sec=time.perf_counter() - t0)
            for n, net in nets.items():
                t0 = time.perf_counter(); outs[n] = net(g); info[n] = dict(sec=time.perf_counter() - t0)
            # save PNGs and get CER (Tesseract) for each
            for m, img in outs.items():
                Image.fromarray(u8(img)).save(os.path.join(d, f"{sid}__{m}.png"))
                c = M.cer(gt[sid], M.ocr(img))
                info.setdefault(m, {})["cer"] = c
                rows.append(dict(text=sid, condition=cond, motion=r.blur_model, L_true=r.length_px,
                                 theta_true=exp02.true_angle(r.blur_model) if r.blur_model != "shake" else np.nan,
                                 sigma=r.sigma, method=m, cer=c, **{k: v for k, v in info[m].items() if k != "cer"}))
            # A4 page: 8 panels (7 images + 2D cepstrum of the blurred one)
            fig = plt.figure(figsize=(8.27, 11.69))
            fig.suptitle(f"{sid}  |  motion: {MOTION[r.blur_model]}  |  L = {r.length_px:g} px  |  noise σ = {r.sigma:g}"
                         f"\nα² chosen on validation: 1D = {a1:g}, 2D = {a2:g}", fontsize=10, family="serif")
            panels = ["sharp", "blurred", "pinv_oracle", "pinv_blind", "pinv2d_blind", "dgv2_mobilenet",
                      "dgv2_inception", "cepstrum"]
            for k, m in enumerate(panels):
                ax = fig.add_subplot(4, 2, k + 1)
                if m == "cepstrum":
                    # central 81x81 of the cepstrum, circles at the found peak (+-L cos, -+L sin);
                    # sign flipped (-crop), so the negative peak looks bright here
                    cp = cepstrum2d(g)
                    h, w = cp.shape
                    R = 40
                    crop = cp[h // 2 - R:h // 2 + R + 1, w // 2 - R:w // 2 + R + 1]
                    lo, hi = np.percentile(-crop, [1, 99.7])
                    ax.imshow(-crop, cmap="gray", extent=(-R, R, R, -R), vmin=lo, vmax=hi)
                    dx, dy = L2 * np.cos(np.deg2rad(th2)), -L2 * np.sin(np.deg2rad(th2))
                    ax.plot([dx, -dx], [dy, -dy], "o", mfc="none", mec="k", ms=9)
                    ax.set_title(f"2D cepstrum of the input (dark = negative)\nfound L = {L2:.1f} px, θ = {th2:.1f}°",
                                 fontsize=8, family="serif")
                    ax.tick_params(labelsize=6)
                    continue
                if m not in outs:
                    ax.axis("off")
                    ax.set_title(f"{TITLE[m]}: not available", fontsize=8)
                    continue
                ax.imshow(np.clip(outs[m], 0, 255), cmap="gray", vmin=0, vmax=255)
                ax.set_xticks([]), ax.set_yticks([])
                s = f"{TITLE[m]}\nCER = {100 * info[m]['cer']:.1f} %"
                if "L" in info[m]:
                    s += f",  L = {info[m]['L']:.1f} px"
                if "theta" in info[m]:
                    s += f", θ = {info[m]['theta']:.1f}°"
                if "alpha2" in info[m]:
                    s += f", α² = {info[m]['alpha2']:g}"
                if "sec" in info[m] and m not in ("sharp", "blurred"):
                    s += f"\ntime {1000 * info[m]['sec']:.0f} ms"
                ax.set_title(s, fontsize=8, family="serif")
            fig.tight_layout(rect=(0, 0, 1, 0.95))
            pdf.savefig(fig)
            plt.close(fig)
            print(f"  {sid} {cond}: " + ", ".join(f"{m} {100 * info[m]['cer']:.0f}%" for m in outs if m != "sharp"))
        pdf.close()
    pd.DataFrame(rows).to_csv(os.path.join(a.out, "atlas_params.csv"), index=False)
    make_examples_figure(a.out, a.ids.split(",")[0])


# 3 conditions (horizontal L=20, 8 deg L=20, shake L=10; sigma=2) x 6 columns.
# Only the left half of each image is shown (bigger letters); CER is for the whole image.
def make_examples_figure(out, sid):
    """Examples figure. Also works on its own: python 05_atlas.py --fig-only --out atlas_out"""
    rows = pd.read_csv(os.path.join(out, "atlas_params.csv"))
    picks = [("sinc_L20_s2", {"en": "Horizontal,\nL = 20 px", "uk": "Горизонтальний,\nL = 20 пікс."}),
             ("angle8_L20_s2", {"en": "At 8°,\nL = 20 px", "uk": "Під кутом 8°,\nL = 20 пікс."}),
             ("shake_L10_s2", {"en": "Hand shake,\nL = 10 px", "uk": "Тремтіння рук,\nL = 10 пікс."})]
    cols = ["sharp", "blurred", "pinv_blind", "pinv2d_blind", "dgv2_mobilenet", "dgv2_inception"]
    head = {"en": ["Sharp", "Blurred", "Pseudo-inverse 1D", "Pseudo-inverse 2D", "DeblurGAN-v2\nMobileNet",
                   "DeblurGAN-v2\nInception"],
            "uk": ["Різке", "Розмите", "Псевдоінверсний 1D", "Псевдоінверсний 2D", "DeblurGAN-v2\nMobileNet",
                   "DeblurGAN-v2\nInception"]}
    p = rows
    for lang in ("en", "uk"):
        fig, axes = plt.subplots(len(picks), len(cols), figsize=(7.0, 0.82 * len(picks) + 0.45))
        for i, (cond, lab) in enumerate(picks):
            for j, m in enumerate(cols):
                ax = axes[i, j]
                fp = os.path.join(out, "atlas", cond, f"{sid}__{m}.png")
                if os.path.exists(fp):
                    im = np.array(Image.open(fp))
                    ax.imshow(im[:, : im.shape[1] // 2], cmap="gray", vmin=0, vmax=255)
                    c = p[(p.condition == cond) & (p.method == m) & (p.text == sid)].cer
                    if len(c) and m != "sharp":
                        ax.set_xlabel(f"CER {100 * c.iloc[0]:.0f} %".replace(".", ",") if lang == "uk"
                                      else f"CER {100 * c.iloc[0]:.0f} %", fontsize=6.5, labelpad=1)
                ax.set_xticks([]), ax.set_yticks([])
                if i == 0:
                    ax.set_title(head[lang][j], fontsize=7, family="serif")
                if j == 0:
                    ax.set_ylabel(lab[lang], fontsize=6.5, family="serif")
        fig.tight_layout(pad=0.3, h_pad=0.4, w_pad=0.3)
        fig.savefig(os.path.join(out, f"fig_examples_{lang}.jpg"), dpi=300, pil_kwargs={"quality": 95})
        fig.savefig(os.path.join(out, f"fig_examples_{lang}.png"), dpi=200)
        plt.close(fig)
    print("done:", out)


if __name__ == "__main__":
    main()

`scripts/06_fig_cer_vs_length.py` - Fig. 1

In [ ]:
%%writefile scripts/06_fig_cer_vs_length.py
"""Fig. 1 of the paper: CER vs blur length for three kinds of motion.
Made from results/quality.csv; black and white, JPG 300 dpi.

  python scripts/06_fig_cer_vs_length.py --res results --lang en --out results/paper_fig1_en.jpg
"""
import argparse

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import pandas as pd  # noqa: E402

T = {"en": dict(groups={"horiz": "(a) Straight horizontal motion", "angle8": "(b) Straight motion at 8°",
                        "shake": "(c) Hand shake"},
                x="Blur length L, px", y="Character error rate, %",
                m={"blurred": "No restoration", "pinv_blind": "Pseudo-inverse 1D", "pinv2d_blind": "Pseudo-inverse 2D",
                   "dgv2_mobilenet": "DeblurGAN-v2 MobileNet", "dgv2_inception": "DeblurGAN-v2 Inception"}),
     "uk": dict(groups={"horiz": "(а) Прямий горизонтальний рух", "angle8": "(б) Прямий рух під кутом 8°",
                        "shake": "(в) Тремтіння рук"},
                x="Довжина змазу L, пікс.", y="Частка помилкових символів, %",
                m={"blurred": "Без відновлення", "pinv_blind": "Псевдоінверсний 1D",
                   "pinv2d_blind": "Псевдоінверсний 2D", "dgv2_mobilenet": "DeblurGAN-v2 MobileNet",
                   "dgv2_inception": "DeblurGAN-v2 Inception"})}
STYLE = {"blurred": ("0.6", ":", "x"), "pinv_blind": ("0.0", "-", "o"), "pinv2d_blind": ("0.0", "--", "v"),
         "dgv2_mobilenet": ("0.45", "-.", "^"), "dgv2_inception": ("0.3", "-", "D")}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--res", default="results")
    ap.add_argument("--lang", default="en")
    ap.add_argument("--out", default="paper_fig1.jpg")
    ap.add_argument("--size", default="6.6,2.3", help="width,height in inches")
    a = ap.parse_args()
    t = T[a.lang]
    q = pd.read_csv(f"{a.res}/quality.csv")
    # sinc and box are both horizontal motion -> one panel; angle8 and shake separate
    q["grp"] = q.blur_model.map({"sinc": "horiz", "box": "horiz", "angle8": "angle8", "shake": "shake"})
    plt.rcParams.update({"font.family": "serif", "font.size": 7.5})
    fig, axes = plt.subplots(1, 3, figsize=tuple(float(v) for v in a.size.split(",")), sharey=True)
    for ax, g in zip(axes, ["horiz", "angle8", "shake"]):
        # mean CER (%) per method and L, over 30 texts and 4 noise levels
        d = q[q.grp == g].groupby(["method", "length_px"]).cer.mean().reset_index()
        for m, (c, ls, mk) in STYLE.items():
            dm = d[d.method == m]
            ax.plot(dm.length_px, 100 * dm.cer, color=c, ls=ls, marker=mk, ms=3.5, lw=1.1, label=t["m"][m])
        ax.set_title(t["groups"][g], fontsize=7.5)
        ax.set_xticks([10, 20, 32])
        ax.set_xlabel(t["x"])
        ax.grid(alpha=.3)
        ax.set_ylim(0, 100)
    axes[0].set_ylabel(t["y"])
    h, lab = axes[0].get_legend_handles_labels()
    fig.legend(h, lab, loc="lower center", ncol=3, frameon=False, fontsize=6.8, bbox_to_anchor=(0.5, -0.01))
    fig.tight_layout(rect=(0, 0.15, 1, 1), w_pad=0.6)
    fig.savefig(a.out, dpi=300, pil_kwargs={"quality": 95})
    fig.savefig(a.out.rsplit(".", 1)[0] + ".png", dpi=200)


if __name__ == "__main__":
    main()

`scripts/07_fig_cepstrum.py` - extra figure with cepstra

In [ ]:
%%writefile scripts/07_fig_cepstrum.py
"""Extra figure (not in the paper): centre of the 2D cepstrum of a blurred image
for three kinds of motion.

Uses real blurred images from the atlas (text test_000, sigma = 2) and the same cepstrum
function as the experiment. Straight motion gives a pair of clear negative (dark) peaks at
distance L from the centre along the motion; hand shake gives a blurry spot with no single peak.

Run after step 5:
  python scripts/07_fig_cepstrum.py --atlas atlas_out --id test_000 --lang en --out results/fig_cepstrum_en.jpg
"""
import argparse
import os
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
from PIL import Image  # noqa: E402

sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))
from deblur.classical2d import cepstrum2d, estimate_motion_2d  # noqa: E402

PICKS = [("sinc_L20_s2", {"uk": "(а) Горизонтальний рух, L = 20 пікс.", "en": "(a) Horizontal, L = 20 px"}),
         ("angle8_L20_s2", {"uk": "(б) Під кутом 8°, L = 20 пікс.", "en": "(b) At 8°, L = 20 px"}),
         ("shake_L20_s2", {"uk": "(в) Тремтіння рук, L = 20 пікс.", "en": "(c) Hand shake, L = 20 px"})]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--atlas", default="atlas_out")
    ap.add_argument("--id", default="test_000")
    ap.add_argument("--lang", default="uk")
    ap.add_argument("--half", type=int, default=32, help="half size of the shown area, px")
    ap.add_argument("--out", default="fig_cepstrum.jpg")
    a = ap.parse_args()
    lab = {"uk": "оцінка: L = {L:.1f} пікс., θ = {t:.1f}°", "en": "estimate: L = {L:.1f} px, θ = {t:.1f}°"}
    fig, axes = plt.subplots(1, len(PICKS), figsize=(6.6, 2.45))
    for ax, (cond, title) in zip(axes, PICKS):
        g = np.array(Image.open(os.path.join(a.atlas, "atlas", cond, f"{a.id}__blurred.png")).convert("L"),
                     dtype=np.float64)
        # same cepstrum as in the experiment (deblur/classical2d.py)
        c = cepstrum2d(g)
        h, w = c.shape
        cy, cx, k = h // 2, w // 2, a.half
        crop = c[cy - k:cy + k + 1, cx - k:cx + k + 1].copy()
        # central 65x65; the centre value is huge and would wash everything out, so hide it
        crop[k, k] = np.nan
        lo, hi = np.nanpercentile(crop, [0.5, 99.5])
        ax.imshow(crop, cmap="gray", vmin=lo, vmax=hi, extent=(-k - .5, k + .5, k + .5, -k - .5))
        # same L and theta estimate as in the experiment, shown under the panel
        L, t = estimate_motion_2d(g)
        if t > 90:
            t -= 180.0                            # theta and theta - 180 are the same line
        t = 0.0 if abs(t) < 0.05 else t
        ax.set_title(title[a.lang], fontsize=8, family="serif")
        txt = lab[a.lang].format(L=L, t=t)
        if a.lang == "uk":
            txt = txt.replace(f"{L:.1f}", f"{L:.1f}".replace(".", ",")).replace(f"{t:.1f}", f"{t:.1f}".replace(".", ","))
        ax.set_xlabel(txt.replace("-", "−"), fontsize=7.5, family="serif")
        ax.set_xticks([-k, 0, k]), ax.set_yticks([-k, 0, k])
        ax.tick_params(labelsize=6.5)
    fig.tight_layout(pad=0.3, w_pad=0.6)
    fig.savefig(a.out, dpi=300, pil_kwargs={"quality": 95})
    print("saved:", a.out)


if __name__ == "__main__":
    main()

`scripts/08_paper_numbers.py` - all numbers used in the paper

In [ ]:
%%writefile scripts/08_paper_numbers.py
"""Step 8: every number used in the paper.

Only reads the result files (quality.csv, efficiency.csv, stats_paired.csv)
and prints each number with a short note on how it was obtained.

Run after step 4:  python scripts/08_paper_numbers.py --res results
"""
import argparse
import os

import numpy as np
import pandas as pd

METHODS = ["blurred", "pinv_blind", "pinv2d_blind", "dgv2_mobilenet", "dgv2_inception"]
NAMES = {"blurred": "No restoration", "pinv_blind": "Pseudo-inverse 1D", "pinv2d_blind": "Pseudo-inverse 2D",
         "dgv2_mobilenet": "DeblurGAN-v2 MobileNet", "dgv2_inception": "DeblurGAN-v2 Inception"}
# sinc and box are both horizontal motion
GROUP = {"sinc": "horizontal", "box": "horizontal", "angle8": "angle 8°", "shake": "hand shake"}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--res", default="results")
    a = ap.parse_args()
    q = pd.read_csv(os.path.join(a.res, "quality.csv"))
    eff = pd.read_csv(os.path.join(a.res, "efficiency.csv"))
    q["grp"] = q.blur_model.map(GROUP)
    global METHODS
    METHODS = [m for m in METHODS if m in set(q.method)]   # in case a network did not load
    pd.set_option("display.width", 200)

    print("=" * 100)
    print("0. Check: Tesseract CER on the sharp images (mean over 30 texts), %:",
          round(100 * q.drop_duplicates("sharp_id").cer_sharp.mean(), 2))

    print("=" * 100)
    print("TABLE 1. CER, % - mean over all images of the group (30 texts x 3 lengths x 4 noise levels;"
          "\n horizontal also x 2 models). Images per group:",
          q[q.method == "blurred"].groupby("grp").size().to_dict())
    t1 = (100 * q[q.method.isin(METHODS)].groupby(["method", "grp"]).cer.mean()).unstack().loc[METHODS]
    t1.index = [NAMES[m] for m in METHODS]
    print(t1[["horizontal", "angle 8°", "hand shake"]].round(1).to_string())
    print("\nTime (ms per image) and energy (J per image, whole machine) from efficiency.csv:")
    print(eff[["method", "device", "ms_per_img_mean", "gross_J_per_img", "cpu_model", "gpu_model",
               "cpu_tracking"]].round(2).to_string(index=False))

    print("=" * 100)
    print("TABLE 2. CER, %, horizontal motion, by noise sigma:")
    h = q[(q.grp == "horizontal") & q.method.isin(METHODS)]
    t2 = (100 * h.groupby(["method", "sigma"]).cer.mean()).unstack().loc[METHODS]
    t2.index = [NAMES[m] for m in METHODS]
    print(t2.round(2).to_string())

    print("=" * 100)
    print("BLUR PARAMETER ERRORS (median |estimate - true|):")
    b = q[q.method == "pinv_blind"]
    e1 = (b.beta_est * 512 / np.pi - b.length_px).abs()       # 1D: beta -> L in px
    print("  1D, L, px:", e1.groupby(b.blur_model).median().round(3).to_dict())
    d = q[q.method == "pinv2d_blind"]
    eL = (d.len2d_est - d.length_px).abs()
    ang = d.angle2d_est % 180
    true = np.where(d.blur_model == "angle8", 8.0, 0.0)
    eA = np.minimum(np.abs(ang - true), 180 - np.abs(ang - true))
    print("  2D, L, px:", eL.groupby(d.blur_model).median().round(3).to_dict())
    print("  2D, theta, deg (means nothing for shake):", pd.Series(eA, index=d.index).groupby(d.blur_model)
          .median().round(3).to_dict())

    print("=" * 100)
    print("HAND SHAKE by length L, CER, % (Fig. 1c):")
    s = q[(q.blur_model == "shake") & q.method.isin(METHODS)]
    print((100 * s.groupby(["length_px", "method"]).cer.mean()).unstack()[METHODS].round(1).to_string())
    per_text = s.groupby(["sharp_id", "method"]).cer.mean().unstack()   # 30 texts x method (mean over 12 conditions)
    if "dgv2_inception" in per_text:
        print("  texts where Inception beats the 2D filter:",
              int((per_text.dgv2_inception < per_text.pinv2d_blind).sum()), "of", len(per_text))

    print("=" * 100)
    print("PAIRED COMPARISONS BY TEXT (stats_paired.csv, CER): share of texts where the first method is better, and p:")
    sp = pd.read_csv(os.path.join(a.res, "stats_paired.csv"))
    sp = sp[(sp.metric == "cer") & ~sp["compare"].str.contains("oracle")]
    print(sp[["group", "compare", "mean_a", "mean_b", "share_a_better", "p_wilcoxon"]].round(4).to_string(index=False))

    print("=" * 100)
    print("SSIM (for reference):")
    print(q[q.method.isin(METHODS)].groupby(["grp", "method"]).ssim.mean().unstack()[METHODS].round(3).to_string())


if __name__ == "__main__":
    main()

## 3. Dependencies
Tesseract (system package), CodeCarbon for energy, pytesseract, rapidfuzz for CER, gdown for the weights,
pretrainedmodels for the Inception-ResNet-v2 backbone, PyWavelets for `estimate_sigma`.
The rest are needed by the DeblurGAN-v2 repo. The last lines print the hardware.

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y tesseract-ocr > /dev/null
!pip install -q codecarbon pytesseract rapidfuzz torchsummary gdown remotezip pillow-heif tabulate PyWavelets
!pip install -q --no-build-isolation pretrainedmodels
!tesseract --version | head -1
import torch, platform
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
!lscpu | grep 'Model name'

## 4. DeblurGAN-v2: authors' code and weights
Clone the authors' repo and download their trained weights (we do not train anything).
If Google Drive refuses (download limit), the cell looks in attached datasets (`/kaggle/input`),
and for Inception tries a mirror on Hugging Face.

In [ ]:
import glob, os, shutil, subprocess
![ -d DeblurGANv2 ] || git clone -q https://github.com/VITA-Group/DeblurGANv2
os.makedirs('weights', exist_ok=True)
# authors' Google Drive first, for Inception also a mirror
SRC = {'fpn_mobilenet.h5': ['gdown:1JhnT4BBeKBBSLqTo6UsJ13HeBXevarrU'],
       'fpn_inception.h5': ['gdown:1UXcsRVW-6KF23_TNzxw-xC0SzaMfXOaR',
                            'url:https://huggingface.co/sayed99/Image-Debluring/resolve/main/DeblurGanV2/fpn_inception.h5']}

def loadable(p):
    # the file really opens in torch (not an html error page)
    try:
        import torch
        ck = torch.load(p, map_location='cpu', weights_only=False)
        return isinstance(ck, dict) and len(ck) > 0
    except Exception as e:
        print('   cannot read', p, e); return False

for name, sources in SRC.items():
    dst = f'weights/{name}'
    if os.path.exists(dst) and loadable(dst):
        continue
    local = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if local:
        shutil.copy(local[0], dst)
    for s in sources:
        if os.path.exists(dst) and loadable(dst):
            break
        kind, ref = s.split(':', 1)
        cmd = ['gdown', '-q', ref, '-O', dst] if kind == 'gdown' else ['wget', '-q', '-O', dst, ref]
        subprocess.run(cmd)
    print(name, 'OK' if os.path.exists(dst) and loadable(dst) else 'missing, this network will be skipped')
!ls -la weights

## 5. Data (about 1 min)
Folder `data`: `sharp/`, `degraded/`, `meta.csv`, `gt_text.json`.
Everything uses seed 2026 and fixed fonts, so the texts are the same every time:
md5 of `gt_text.json` should be `3cdc024f…`.

In [ ]:
if RUN_SYNTHETIC:
    !python scripts/01_make_dataset.py --out data --n-test {N_TEST} --n-val {N_VAL} --corpus texts/holmes_scenes.txt
    import hashlib
    h = hashlib.md5(open('data/gt_text.json', 'rb').read()).hexdigest()
    if (N_TEST, N_VAL) == (30, 5):
        print('md5 of the texts:', h, 'OK' if h == '3cdc024f2f7341fd61d69f950ce01d22' else 'DIFFERENT from our run')

## 6. Main experiment (about 55 min on T4)
1. α² on the 5 validation texts: for each noise level the value from 1e-8 … 1 with the best PSNR (1D and 2D separately)
2. all 1440 test images × 6 methods → Tesseract → CER (and WER, PSNR, SSIM, estimated L and θ);
   on test the noise level is estimated automatically
3. time and energy on the 30 images of `sinc_L32_s2`

Main output: `results/quality.csv`, one row per image × method.

In [ ]:
if RUN_SYNTHETIC:
    !python scripts/02_run_experiment.py --data data --out results --energy-seconds 30

## 7. Statistics and extra figures
`04_stats.py`: one CER value per text and method (mean over conditions), bootstrap 95 % CI, paired Wilcoxon test.
`03_make_figures.py`: figures to look through the results and `results/table_main.md`.

In [ ]:
if RUN_SYNTHETIC:
    !python scripts/04_stats.py --synthetic results
    !python scripts/03_make_figures.py --data data --res results
    print(open('results/table_main.md').read())
    from IPython.display import Image, display
    for f in ['fig3_quality_vs_energy', 'fig1_cer_vs_length', 'fig4_examples']:
        p = f'results/figures/{f}.png'
        if os.path.exists(p):
            display(Image(p, width=700))

## 8. Atlas and Fig. 2 (about 6 min)
For two test texts, one PDF page per condition: sharp, blurred, every method, 2D cepstrum,
estimated L and θ, α², CER and time. Also `fig_examples_en.jpg` (Fig. 2 of the paper).

In [ ]:
if RUN_SYNTHETIC:
    !python scripts/05_atlas.py --data data --out atlas_out --ids test_000,test_001
    from IPython.display import Image, display
    display(Image('atlas_out/fig_examples_en.png', width=800))

## 9. Fig. 1 and the cepstrum figure
Fig. 1 from `results/quality.csv`; cepstra of real blurred images from the atlas (test_000, σ = 2).

In [ ]:
if RUN_SYNTHETIC:
    for lang in ('en', 'uk'):
        !python scripts/06_fig_cer_vs_length.py --res results --lang {lang} --out results/paper_fig1_{lang}.jpg
        !python scripts/07_fig_cepstrum.py --atlas atlas_out --id test_000 --lang {lang} --out results/fig_cepstrum_{lang}.jpg
    from IPython.display import Image, display
    display(Image('results/paper_fig1_en.png', width=800))
    display(Image('results/fig_cepstrum_en.jpg', width=800))

## 10. All numbers used in the paper
Table 1, CER vs noise, errors of L and θ, hand shake by length, paired comparisons, SSIM.

In [ ]:
if RUN_SYNTHETIC:
    !python scripts/08_paper_numbers.py --res results | tee results/paper_numbers.txt

## 11. Zip for the Output tab
Restored images (`results/restored`) are left out, they are too big.

In [ ]:
%cd {WORK}
!rm -f /kaggle/working/results_meics.zip
!zip -qr /kaggle/working/results_meics.zip results atlas_out -x "results/restored/*" 2>/dev/null
!ls -la /kaggle/working/results_meics.zip